# 09 · Commenti dei lettori agli articoli ICE del NYT

Questo notebook analizza i commenti che i lettori del New York Times hanno scritto sotto gli articoli del corpus ICE. I commenti sono una reazione del pubblico, ma prodotta dentro lo spazio del giornale: il NYT decide su quali articoli aprirli, li modera prima della pubblicazione. Prima di confrontarne il tono con quello degli articoli (notebook 11) serve sapere dove si concentrano, chi li scrive e come si sviluppa la discussione.

**Provenienza dei dati.** I commenti sono stati raccolti nel notebook 01 §6 (dettagli in `docs/nyt_comments.md`). La Community API ufficiale del NYT non è più disponibile: ogni suo endpoint risponde 404, con o senza chiave. La raccolta usa quindi l'endpoint interno con cui il sito carica i commenti, che non è documentato e va dichiarato come limite nel report. Su 2.093 articoli ICE, 687 hanno commenti; i commenti e le risposte raccolti sono 286.399, il 99,9% di quelli dichiarati dal NYT.

**Struttura.**
- §1 caricamento e pseudonimizzazione degli utenti;
- §2 quali articoli ricevono commenti, per anno e per sezione;
- §3 quanto i commenti si concentrano su pochi articoli;
- §4 conflittualità dei thread e burst del dibattito (algoritmo di Kleinberg);
- §5 temi dei commenti (BERTopic);
- §5 note per il report.

**Input**
- `data/processed/nyt/comments_ice.parquet`: un commento o una risposta per riga (286.399).
- `data/processed/nyt/comments_ice_articles.parquet`: un articolo ICE per riga, con i totali dichiarati dal NYT (2.093).
- `data/processed/nyt/articles_ice.parquet`: metadati degli articoli.

**Dati personali.** `user_id` e i nomi dei lettori non escono dal §1: da lì in poi ogni utente è identificato solo dallo pseudonimo `user_key`, e tutte le tabelle e i grafici scritti dal notebook usano solo quello. Il procedimento è descritto nel §1.

**Fuori da questo notebook**: il sentiment dei commenti, misurato nel notebook 11 con gli stessi modelli usati per gli articoli.

## 0. Setup

Percorsi di input e output, logging, sale per la pseudonimizzazione e funzioni di supporto usate in più sezioni.

- **Sale.** `COMMENTS_PSEUDONYM_SALT` si legge da `.env`; se manca, il notebook si ferma prima di caricare i dati. Il valore non va mai cambiato: con un sale diverso tutti gli pseudonimi cambiano e le tabelle prodotte in esecuzioni diverse non sono più confrontabili tra loro.
- **Periodi.** `get_periods` e `assign_period` riproducono i quattro periodi dei notebook 02-08 (2003-2016, 2017-2020, 2021-2024, 2025-presente). Un commento eredita il periodo dall'anno di pubblicazione del suo articolo, non dalla propria data, così un thread resta intero nello stesso periodo. Il caso di un articolo di fine dicembre commentato a gennaio è raro, perché la discussione si esaurisce quasi sempre in pochi giorni.
- **Misure di concentrazione** (usate nel §3). `gini` calcola l'indice di Gini sui valori ordinati in modo crescente, *G* = Σ(2*i* − *n* − 1)·*x<sub>i</sub>* / (*n*·Σ*x*): vale 0 se tutti gli elementi hanno lo stesso valore e si avvicina a 1 se il totale è concentrato su un solo elemento. `top_share(values, f)` restituisce la quota del totale che spetta alla frazione *f* degli elementi più grandi (per esempio *f* = 0,10 è il 10% degli articoli più commentati).
- **Grafici.** Un colore fisso per periodo, uguale in tutte le figure del notebook. `FIG_DIR` (`docs/figures/`) raccoglie le figure destinate alla tesi.

In [ ]:
import hashlib
import logging
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

%matplotlib inline

# Radice del progetto: funziona sia lanciando da notebooks/ sia dalla radice
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "nyt"
PLOTS_DIR = PROCESSED_DIR / "plots"
FIG_DIR = BASE_DIR / "docs" / "figures"  # figure per la tesi

# Input: parsing dei commenti (notebook 01 §6) e degli articoli ICE (notebook 01 §2)
IN_COMMENTS = PROCESSED_DIR / "comments_ice.parquet"
IN_COMMENT_ARTICLES = PROCESSED_DIR / "comments_ice_articles.parquet"
IN_ARTICLES = PROCESSED_DIR / "articles_ice.parquet"

# Output: tutti rigenerabili, nessuno contiene user_id o nomi dei lettori
OUT_COMMENTS_PSEUDO = PROCESSED_DIR / "comments_ice_pseudo.parquet"
OUT_ARTICLE_STATS = PROCESSED_DIR / "comments_article_stats.csv"
OUT_THREAD_METRICS = PROCESSED_DIR / "comments_thread_metrics.csv"
OUT_BURSTS = PROCESSED_DIR / "comments_bursts.csv"
OUT_COMMENT_TOPICS = PROCESSED_DIR / "comments_topics.parquet"
OUT_COMMENT_TOPIC_INFO = PROCESSED_DIR / "comments_topic_info.csv"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("nyt_comments_analysis")

# Il sale non va mai cambiato: un sale diverso cambia tutti gli pseudonimi
load_dotenv(BASE_DIR / ".env")
PSEUDONYM_SALT = os.getenv("COMMENTS_PSEUDONYM_SALT")
if not PSEUDONYM_SALT:
    raise RuntimeError("COMMENTS_PSEUDONYM_SALT non trovato in .env")

# Slot categoriali della palette di riferimento, in ordine fisso: un colore per periodo
PERIOD_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
MAIN_COLOR = PERIOD_COLORS[0]
MUTED_COLOR = "#b8b8b0"

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e6e6e1",
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
})


def get_periods(max_year: int) -> list[tuple[str, int, int]]:
    return [
        ("2003-2016", 2003, 2017),
        ("2017-2020", 2017, 2021),
        ("2021-2024", 2021, 2025),
        ("2025-presente", 2025, max_year + 1),
    ]


def assign_period(years: pd.Series) -> pd.Series:
    periods = get_periods(int(years.max()))
    labels = [label for label, _, _ in periods]
    bins = [periods[0][1]] + [end for _, _, end in periods]
    return pd.cut(years, bins=bins, right=False, labels=labels)


def gini(values) -> float:
    """Indice di Gini (0 = distribuzione uniforme, 1 = tutto su un solo elemento)."""
    # G = sum((2i - n - 1) * x_i) / (n * sum(x)), con x ordinato in modo crescente e i = 1..n
    x = np.sort(np.asarray(values, dtype=float))
    n = len(x)
    if n == 0 or x.sum() == 0:
        return 0.0
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))


def top_share(values, fraction: float) -> float:
    """Quota del totale detenuta dal `fraction` piu' grande degli elementi (es. 0.01 = top 1%)."""
    x = np.sort(np.asarray(values, dtype=float))[::-1]
    k = max(1, int(round(len(x) * fraction)))
    return float(x[:k].sum() / x.sum())


def plot_flow(stages: list[tuple[str, str, str]], title: str, filename: str, height: float = 3.0) -> None:
    """Schema a riquadri in fila (titolo, numero o sintesi, nota), salvato in FIG_DIR per la tesi."""
    from matplotlib.patches import FancyBboxPatch

    n = len(stages)
    fig, ax = plt.subplots(figsize=(2.6 * n + 0.5, height))
    ax.set_xlim(-0.01, 1.01)
    ax.set_ylim(0, 1)
    ax.axis("off")
    gap = 0.035
    box_w = (1 - 0.02 - gap * (n - 1)) / n
    for i, (head, big, note) in enumerate(stages):
        x0 = 0.01 + i * (box_w + gap)
        final = i == n - 1
        ax.add_patch(FancyBboxPatch((x0, 0.06), box_w, 0.86, boxstyle="round,pad=0.005,rounding_size=0.02",
                                    facecolor=MAIN_COLOR if final else "#e8f0fb", edgecolor=MAIN_COLOR, linewidth=1))
        color = "white" if final else "#1a1a1a"
        ax.text(x0 + box_w / 2, 0.83, head, ha="center", va="center", fontsize=9.5, fontweight="bold", color=color)
        ax.text(x0 + box_w / 2, 0.62, big, ha="center", va="center", fontsize=14 if len(big) < 12 else 10,
                fontweight="bold", color=color)
        ax.text(x0 + box_w / 2, 0.28, note, ha="center", va="center", fontsize=7.8, color=color, linespacing=1.35)
        if not final:
            ax.annotate("", xy=(x0 + box_w + gap - 0.003, 0.5), xytext=(x0 + box_w + 0.003, 0.5),
                        arrowprops={"arrowstyle": "->", "color": "#555555"})
    ax.set_title(title, fontsize=11, loc="left")
    fig.savefig(FIG_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


## 1. Caricamento e pseudonimizzazione

I commenti contengono dati personali di lettori reali: l'identificativo dell'account NYT (`user_id`), il nome visualizzato (`user_display_name`), il nome dell'utente a cui si risponde (`parent_user_display_name`) e la località scritta dal lettore (`user_location`). Le analisi non hanno bisogno di sapere chi è un utente. Devono solo riconoscere lo stesso utente in commenti diversi, per ricostruire chi risponde a chi (§4). Questa sezione sostituisce quindi l'identificativo con uno pseudonimo stabile e rimuove i nomi prima di qualsiasi analisi.

**Pseudonimo.** `user_key` = `"u_"` + i primi 12 caratteri esadecimali di SHA-256(sale + `user_id`).
- Il sale è una stringa segreta conservata in `.env`. Senza sale, chi conosce un `user_id` potrebbe calcolarne l'hash e ritrovare l'utente nelle tabelle; con il sale serve anche il segreto.
- Lo stesso `user_id` produce sempre lo stesso `user_key` finché il sale resta invariato.
- 12 caratteri esadecimali sono 48 bit. Con 76.387 utenti la probabilità che due utenti ricevano lo stesso pseudonimo è circa 1 su 100.000; il codice lo verifica comunque (gli pseudonimi distinti devono essere quanti gli `user_id` distinti) e si ferma in caso di collisione.

**Chi risponde a chi.** `parent_user_key` è lo pseudonimo dell'autore del commento a cui una risposta è rivolta. Si ricava agganciando `parent_id` al `comment_id` del commento padre, e non dal campo `parent_user_display_name`: i nomi visualizzati non sono univoci (due lettori possono firmarsi allo stesso modo) e sono dati personali. 182 risposte hanno un padre che non compare tra i commenti raccolti, verosimilmente un commento rimosso dopo la pubblicazione; per queste `parent_user_key` resta vuoto.

**Periodo.** Ogni commento riceve l'anno e il periodo dell'articolo a cui appartiene (§0).

**Output.** `comments_ice_pseudo.parquet` è la stessa tabella senza `user_id` e senza nomi, ed è la versione letta da tutte le sezioni successive e dal notebook 11. Pseudonimizzato non vuol dire anonimo: chi possiede il sale può ricollegare gli pseudonimi agli account; il testo dei commenti può contenere nomi e menzioni (`@nome`), che vengono ripuliti solo nel §5 per il topic modeling; `user_location` resta come testo libero e va aggregata prima di pubblicare qualsiasi tabella che la contenga.

In [ ]:
def pseudonymize(user_ids: pd.Series, salt: str) -> pd.Series:
    """user_id -> 'u_' + primi 12 caratteri esadecimali di sha256(sale + id). Stabile a parita' di sale."""
    # 12 caratteri esadecimali = 48 bit: con ~76 mila utenti una collisione ha probabilita' ~1e-5,
    # e il controllo in load_comments la esclude comunque
    return user_ids.map(lambda uid: "u_" + hashlib.sha256(f"{salt}{uid}".encode()).hexdigest()[:12])


def load_comments() -> pd.DataFrame:
    df = pd.read_parquet(IN_COMMENTS)
    df["user_key"] = pseudonymize(df["user_id"], PSEUDONYM_SALT)
    if df["user_key"].nunique() != df["user_id"].nunique():
        raise RuntimeError("Collisione di pseudonimi: aumentare la lunghezza dell'hash")

    # Pseudonimo dell'autore del commento padre, da parent_id e non dal nome visualizzato (non univoco)
    parent_keys = df.set_index("comment_id")["user_key"]
    df["parent_user_key"] = df["parent_id"].map(parent_keys)

    # Da qui in poi nessuna colonna identifica direttamente il lettore
    df = df.drop(columns=["user_id", "user_display_name", "parent_user_display_name"])
    # Anno e periodo dell'articolo, non del commento: un thread resta tutto nel periodo del suo articolo
    df["year"] = df["article_pub_date"].dt.year
    df["period"] = assign_period(df["year"])
    return df


def load_comment_articles() -> pd.DataFrame:
    df = pd.read_parquet(IN_COMMENT_ARTICLES)
    df["year"] = df["article_pub_date"].dt.year
    df["period"] = assign_period(df["year"])
    return df


comments = load_comments()
comment_articles = load_comment_articles()

# Risposte il cui padre non e' tra i commenti raccolti (verosimilmente rimosso): parent_user_key vuoto
missing_parent = comments["is_reply"] & comments["parent_user_key"].isna()
log.info("Commenti: %d, utenti: %d, articoli: %d", len(comments), comments["user_key"].nunique(), len(comment_articles))
log.info("Risposte senza commento padre tra i raccolti: %d", missing_parent.sum())

# Versione condivisibile della tabella, letta dalle sezioni successive e dal notebook 11
comments.to_parquet(OUT_COMMENTS_PSEUDO, index=False)
log.info("Scritto %s", OUT_COMMENTS_PSEUDO.relative_to(BASE_DIR))

In [ ]:
from matplotlib.patches import FancyBboxPatch

# Schema della pseudonimizzazione per la tesi; i numeri sono letti dalle variabili del §1
n_users = comments["user_key"].nunique()
n_missing_parent = int(missing_parent.sum())

fig, ax = plt.subplots(figsize=(12, 4.2))
ax.set_xlim(-0.01, 1)
ax.set_ylim(0, 1)
ax.axis("off")


def box(x, y, w, h, title, body, highlight=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.005,rounding_size=0.015",
                                facecolor=MAIN_COLOR if highlight else "#e8f0fb", edgecolor=MAIN_COLOR, linewidth=1))
    color = "white" if highlight else "#1a1a1a"
    ax.text(x + w / 2, y + h - 0.06, title, ha="center", va="top", fontsize=10, fontweight="bold", color=color)
    ax.text(x + w / 2, y + (h - 0.09) / 2, body, ha="center", va="center", fontsize=8.5, color=color, linespacing=1.4)


def arrow(xy_from, xy_to, label=None, label_xy=None):
    ax.annotate("", xy=xy_to, xytext=xy_from, arrowprops={"arrowstyle": "->", "color": "#555555", "linewidth": 1})
    if label:
        ax.text(*label_xy, label, ha="center", va="center", fontsize=8, color="#333333", style="italic")


box(0.00, 0.22, 0.21, 0.56, "comments_ice.parquet",
    f"{len(comments):,} comments\n\nuser_id\nuser_display_name\nparent_user_display_name\nparent_id")
box(0.31, 0.78, 0.21, 0.21, ".env", "COMMENTS_PSEUDONYM_SALT")
box(0.31, 0.44, 0.21, 0.28, "user_key", f"u_ + sha256(salt + user_id)[:12]\n{n_users:,} keys, 0 collisions")
box(0.31, 0.02, 0.21, 0.30, "parent_user_key", f"user_key of the comment\nparent_id points to\n{n_missing_parent} replies: parent not collected")
box(0.62, 0.22, 0.32, 0.56, "comments_ice_pseudo.parquet",
    "user_id and display names dropped\n\nuser_location kept (free text:\naggregate before publishing)\n\nread by §2-§5 and notebook 11", highlight=True)

arrow((0.21, 0.58), (0.31, 0.58), "hash", (0.26, 0.62))
arrow((0.415, 0.78), (0.415, 0.72), "salt", (0.445, 0.75))
arrow((0.21, 0.30), (0.31, 0.17), "join on\nparent_id", (0.245, 0.17))
arrow((0.415, 0.44), (0.415, 0.32), "key of\nthe parent", (0.455, 0.38))
arrow((0.52, 0.58), (0.62, 0.55))
arrow((0.52, 0.17), (0.62, 0.40))
ax.set_title("Pseudonymization of NYT reader comments (§1)", fontsize=11, loc="left")
fig.savefig(FIG_DIR / "comments_pseudonymization.png", dpi=200, bbox_inches="tight")
plt.show()


**Come leggere lo schema.** A sinistra la tabella prodotta dal parsing del notebook 01: l'hash con il sale genera `user_key`; l'aggancio tramite `parent_id` assegna a ogni risposta lo pseudonimo dell'autore del commento padre. La figura è salvata in `docs/figures/comments_pseudonymization.png` per la tesi.

## 2. Quali articoli hanno commenti

Il NYT apre i commenti solo su una parte degli articoli, per scelta editoriale. Prima di confrontare il volume dei commenti tra anni o tra sezioni bisogna quindi sapere quanti articoli erano commentabili: un anno con più commenti può avere semplicemente più articoli aperti, non lettori più coinvolti. La sezione separa i due effetti con due misure:

- la **quota di articoli con commenti**, che dipende dalle scelte della redazione;
- i **commenti per articolo commentato**, che misurano la risposta dei lettori dove la possibilità di commentare c'era.

**Definizione.** `has_comments` vale `True` se il NYT dichiara almeno un commento per l'articolo. L'endpoint non dice se i commenti erano aperti, quindi un articolo con i commenti chiusi e uno con i commenti aperti ma senza interventi finiscono nella stessa categoria. Il secondo caso è verosimilmente raro su un tema dibattuto come ICE, ma questi dati non permettono di verificarlo.

**Anno e periodo.** Si usa l'anno di pubblicazione dell'articolo. Il corpus ICE del NYT non contiene articoli prima del 2008, quindi il periodo 2003-2016 copre di fatto il 2008-2016.

In [ ]:
# Articoli, articoli con commenti e commenti per anno: base del grafico
by_year = comment_articles.groupby("year").agg(
    articoli_ice=("article_id", "size"),
    articoli_con_commenti=("has_comments", "sum"),
    commenti=("n_comments_collected", "sum"),
)
by_year["pct_con_commenti"] = (by_year["articoli_con_commenti"] / by_year["articoli_ice"] * 100).round(1)
# replace(0, nan): il 2008 non ha articoli commentati, evita la divisione per zero
by_year["commenti_per_articolo_commentato"] = (
    by_year["commenti"] / by_year["articoli_con_commenti"].replace(0, np.nan)
).round(0)


In [ ]:
# Alto: articoli ICE (grigio) e quanti hanno commenti (blu). Basso: commenti per anno dell'articolo
fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)

axes[0].bar(by_year.index, by_year["articoli_ice"], color=MUTED_COLOR, width=0.8, label="articoli ICE")
axes[0].bar(by_year.index, by_year["articoli_con_commenti"], color=MAIN_COLOR, width=0.8, label="di cui con commenti")
axes[0].set_ylabel("articoli")
axes[0].legend(frameon=False, loc="upper left")
axes[0].set_title("Articoli ICE e articoli con commenti, per anno", loc="left")

axes[1].bar(by_year.index, by_year["commenti"] / 1000, color=MAIN_COLOR, width=0.8)
axes[1].set_ylabel("commenti (migliaia)")
axes[1].set_title("Commenti raccolti, per anno di pubblicazione dell'articolo", loc="left")

# Anni interi sull'asse x (senza, matplotlib mostra 2007.5, 2010.0, ...)
axes[1].xaxis.set_major_locator(plt.MaxNLocator(integer=True))

fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_coverage_by_year.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "comments_coverage_by_year.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

**Come leggere il grafico.** In alto, per anno, gli articoli ICE (grigio) e quanti di essi hanno commenti (blu); in basso i commenti raccolti, attribuiti all'anno di pubblicazione dell'articolo.

- **Il volume segue gli articoli commentabili.** Il pannello inferiore segue le barre blu più di quelle grigie. Le due fasi di maggiore attività, 2017-2019 e 2025-2026, coincidono con i due picchi di copertura del corpus ICE (notebook 01 §5).
- **Squilibrio tra periodi.** Il 2025-presente raccoglie 196.203 commenti, il 68,5% del totale, su 442 articoli commentati (211 nel 2025, 231 nel 2026, contro 27-53 l'anno nel 2017-2019). Ogni misura calcolata sull'insieme dei commenti è dominata da questo periodo: per questo il §4 stima il comportamento "normale" della discussione periodo per periodo e i confronti tra periodi usano quote e mediane.
- **Quota di articoli aperti.** Oscilla senza andamento regolare: dal 10,5% (2010) al 46,2% (2015) fino al 2016, poi tra il 24% e il 43%. Dal 2017 anche la risposta al singolo articolo è più alta (269-664 commenti per articolo commentato, contro 44-283 prima).
- **Date imputate.** Tutti i commenti del 2003-2016 e il 31,2% di quelli del 2017-2020 hanno la data di creazione sostituita con quella di approvazione (notebook 01 §6, migrazione del sistema dei commenti NYT): le misure di tempo su questi commenti sovrastimano il momento della scrittura di qualche decina di minuti.
- Il 2026 è parziale (fino a settembre); gli anni con pochi articoli commentati (3 nel 2023, 4 nel 2016) danno valori instabili.

## 3. Concentrazione per articolo

La sezione misura se i commenti si distribuiscono in modo uniforme tra gli articoli commentati o si concentrano su pochi. La domanda ha conseguenze pratiche. Ogni misura calcolata sull'insieme dei commenti (temi nel §5, sentiment nel notebook 11) pesa ciascun articolo per il numero dei suoi commenti: se pochi articoli raccolgono la maggior parte dei commenti, sono quei pochi articoli a determinare il risultato.

**Misure.** Tutte sono calcolate sui 687 articoli con commenti. Includere i 1.406 articoli senza commenti misurerebbe di nuovo la scelta della redazione di aprire i commenti, già analizzata nel §2.
- **Indice di Gini** (definito nel §0): 0 se ogni articolo ha lo stesso numero di commenti, vicino a 1 se tutti i commenti sono su un solo articolo.
- **Quota del top 1% e del top 10%**: quota dei commenti raccolta dai 7 e dai 69 articoli più commentati.
- **Curva di Lorenz**, per periodo: articoli ordinati dal meno al più commentato sull'asse orizzontale, quota cumulata dei commenti su quello verticale. La diagonale corrisponde alla distribuzione uniforme; più la curva se ne allontana, più i commenti sono concentrati. L'indice di Gini è il doppio dell'area compresa tra la diagonale e la curva.

**Output.** `comments_article_stats.csv`, una riga per articolo commentato: numero di commenti e di risposte, utenti distinti, raccomandazioni totali, commenti selezionati dalla redazione (NYT Picks), risposte dei giornalisti, ore mediane tra la pubblicazione dell'articolo e i commenti, quota di risposte, più data, periodo, sezione, titolo e URL. La tabella è riusata nel §4.

In [ ]:
articles_meta = pd.read_parquet(IN_ARTICLES, columns=["id", "headline", "web_url"]).rename(columns={"id": "article_id"})

# Statistiche per articolo commentato: base del §3 e del §4 (catene e burst)
article_stats = (
    comments.groupby("article_id")
    .agg(
        commenti=("comment_id", "size"),
        risposte=("is_reply", "sum"),
        utenti=("user_key", "nunique"),
        raccomandazioni=("recommendations", "sum"),
        selezioni_editoriali=("editors_selection", "sum"),
        risposte_giornalisti=("is_reporter_reply", "sum"),
        ore_mediane_dalla_pubblicazione=("hours_since_publication", "median"),
    )
    .reset_index()
    .merge(comment_articles[["article_id", "article_pub_date", "period", "section_name"]], on="article_id", how="left")
    .merge(articles_meta, on="article_id", how="left")
    .sort_values("commenti", ascending=False)
)
article_stats["quota_risposte"] = (article_stats["risposte"] / article_stats["commenti"]).round(3)
article_stats.to_csv(OUT_ARTICLE_STATS, index=False)
log.info("Scritto %s", OUT_ARTICLE_STATS.relative_to(BASE_DIR))

# Concentrazione sui soli articoli con commenti: gli zeri misurerebbero la scelta di aprire i commenti (§2)
n = article_stats["commenti"]
pd.Series({
    "articoli con commenti": len(n),
    "mediana commenti/articolo": n.median(),
    "media commenti/articolo": round(n.mean(), 1),
    "massimo": n.max(),
    "gini": round(gini(n), 3),
    "quota top 1% articoli": round(top_share(n, 0.01), 3),
    "quota top 10% articoli": round(top_share(n, 0.10), 3),
})

**Come leggere il riepilogo.** Un articolo commentato riceve in mediana 271 commenti, ma la media è 416,9: la distribuzione è asimmetrica, con pochi articoli molto commentati che alzano la media. Il massimo è 5.847 commenti. I 7 articoli più commentati (top 1%) raccolgono il 6,8% dei commenti; i 69 più commentati (top 10%) il 35,4%. L'indice di Gini è 0,544: la concentrazione è netta, ma la discussione non dipende da una manciata di articoli.

In [ ]:
def lorenz_points(values) -> tuple[np.ndarray, np.ndarray]:
    x = np.sort(np.asarray(values, dtype=float))
    cum = np.concatenate([[0], np.cumsum(x) / x.sum()])
    return np.linspace(0, 1, len(cum)), cum


# Una curva per periodo; la diagonale tratteggiata e' la distribuzione uniforme
fig, ax = plt.subplots(figsize=(5.5, 5))
for color, (label, _, _) in zip(PERIOD_COLORS, get_periods(int(comments["year"].max()))):
    values = article_stats.loc[article_stats["period"] == label, "commenti"]
    if len(values) < 2:
        continue
    px, py = lorenz_points(values)
    ax.plot(px, py, color=color, linewidth=2, label=f"{label} (n={len(values)}, Gini {gini(values):.2f})")
ax.plot([0, 1], [0, 1], color=MUTED_COLOR, linewidth=1, linestyle="--")
ax.set_xlabel("quota cumulata degli articoli (dal meno commentato)")
ax.set_ylabel("quota cumulata dei commenti")
ax.set_title("Concentrazione dei commenti per articolo", loc="left")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_lorenz_by_period.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "comments_lorenz_by_period.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

**Come leggere il grafico.** Una curva di Lorenz per periodo; in legenda il numero di articoli commentati e l'indice di Gini. Una curva più vicina alla diagonale indica commenti distribuiti in modo più uniforme.

- Il 2017-2020 è il periodo più concentrato (Gini 0,62): il solo articolo del 21 febbraio 2017 raccoglie il 9,1% dei commenti del periodo (5.847 su 64.062).
- Il 2025-presente è il meno concentrato (Gini 0,49), nonostante sia il periodo con più commenti: l'attenzione si distribuisce su 442 articoli invece di fissarsi su pochi casi.
- La curva del 2021-2024 poggia su 32 articoli ed è la meno affidabile.

In [ ]:
# Le curve di Lorenz in numeri, per periodo e sul totale; misure di partecipazione per articolo
def concentration_summary(d: pd.DataFrame) -> pd.Series:
    n = d["commenti"]
    return pd.Series({
        "articoli": len(n),
        "mediana_commenti": n.median(),
        "massimo": n.max(),
        "gini": round(gini(n), 3),
        "quota_top_10pct": round(top_share(n, 0.10), 3),
        "mediana_quota_risposte": round(d["quota_risposte"].median(), 3),
        # Commenti per partecipante: vicino a 1 = molti lettori con un commento ciascuno
        "mediana_commenti_per_utente": round((d["commenti"] / d["utenti"]).median(), 2),
        # Il volume di un articolo cresce con il numero di partecipanti o con pochi utenti molto attivi?
        "spearman_commenti_utenti": round(n.corr(d["utenti"], method="spearman"), 3),
    })


concentration = article_stats.groupby("period", observed=True)[["commenti", "utenti", "quota_risposte"]].apply(concentration_summary)
concentration.loc["totale"] = concentration_summary(article_stats)
concentration


**Come leggere la tabella.** Le curve di Lorenz espresse in numeri, una riga per periodo e una per il totale. `mediana_commenti` e `massimo`: commenti per articolo commentato; `gini` e `quota_top_10pct`: concentrazione; `mediana_quota_risposte`: quota tipica di risposte in un articolo; `mediana_commenti_per_utente`: commenti divisi per utenti distinti, per articolo; `spearman_commenti_utenti`: correlazione di rango tra numero di commenti e numero di utenti distinti degli articoli.

- **Risposta crescente.** La mediana dei commenti per articolo commentato sale in ogni periodo: 84,5, 205, 236, 328. Nello stesso tempo la concentrazione scende dal 2017-2020 al 2025-presente (Gini da 0,62 a 0,49, top 10% da 42,4% a 31,0%): nel periodo più recente l'articolo commentato tipico riceve più commenti e la discussione è meno dipendente dai singoli casi.
- **Partecipazione.** La correlazione tra commenti e utenti distinti è 0,99 in tutti i periodi e ogni partecipante scrive in mediana 1,1-1,6 commenti per articolo. Un articolo molto commentato è un articolo commentato da molte persone.
- **Risposte.** Nel 2003-2016 la quota mediana di risposte è 0,28, dal 2017 tra 0,46 e 0,51. Il salto coincide con l'aumento dei commenti per articolo; questi dati non dicono se ne sia la causa.

**Sintesi del §1-§3.** I commenti sono distribuiti in modo molto diseguale nel tempo (due terzi nel 2025-presente) e tra sezioni (Opinion sovrarappresentata); dentro ogni periodo sono concentrati su una parte degli articoli, ma senza dipendere da pochi casi (Gini tra 0,49 e 0,62). Per le analisi successive ne derivano tre accorgimenti: stimare i parametri di riferimento per periodo (§4), confrontare commenti e articoli tenendo conto della sezione (notebook 11) e ricordare che i risultati sull'insieme dei commenti descrivono soprattutto il 2025-presente.

## 4. Conflittualità dei thread e burst del dibattito (Kleinberg)

**Domanda.** I momenti in cui i lettori discutono più a lungo tra loro coincidono con quelli in cui commentano di più? Per ogni periodo si confrontano due serie:

- **volume**: commenti per settimana;
- **catena**: quota giornaliera di risposte in catena, cioè di risposte a una risposta.

Un articolo molto letto può raccogliere migliaia di commenti indipendenti, rivolti all'articolo; un articolo poco letto può ospitare una lunga discussione tra pochi lettori. Per questo le due serie si analizzano separatamente e poi si confrontano (§4.4).

**Profondità della catena.** Il campo `depth` del NYT vale solo 1 o 2, perché l'interfaccia mostra tutte le risposte allo stesso livello. Il campo `parent_id`, invece, punta al commento a cui si risponde davvero. Risalendo `parent_id` si ricostruisce `chain_depth`: 1 = commento all'articolo, 2 = risposta a un commento, 3 = risposta a una risposta, e così via. Le 182 risposte con il padre non raccolto (§1) ricevono profondità 2. Una **risposta in catena** (`deep`) ha `chain_depth ≥ 3`. È una misura strutturale: dice che i lettori discutono tra loro, non che sono in disaccordo (il tono è nel notebook 11).

**Catene solo da aprile 2018.** Fino a ottobre 2017 il NYT registrava ogni risposta come rivolta al commento iniziale del thread; le risposte a risposte compaiono a fine novembre 2017 e si stabilizzano da aprile 2018. Il cambiamento riguarda il sistema dei commenti, non i lettori. La serie della catena parte quindi da `THREADED_REPLIES_START = 2018-04-01` e il 2003-2016 resta fuori.

**Percorso.** §4.1 algoritmo di Kleinberg; §4.2 burst di volume; §4.3 sensibilità ai parametri; §4.4 catena contro volume, la risposta alla domanda.

In [ ]:
# Thread di esempio (dati inventati, solo per illustrare le misure): (id, autore, padre)
EXAMPLE_THREAD = [("C1", "A", None), ("R1", "B", "C1"), ("R2", "A", "R1"), ("R3", "C", "C1"), ("R4", "B", "R2")]
ex = pd.DataFrame(EXAMPLE_THREAD, columns=["id", "author", "parent"])
author_of = dict(zip(ex["id"], ex["author"]))
ex["parent_author"] = ex["parent"].map(author_of)
# Stessa regola di add_chain_depth, applicata al thread di esempio
chain = {}
for row in ex.itertuples():
    chain[row.id] = 1 if pd.isna(row.parent) else chain[row.parent] + 1
ex["chain_depth"] = ex["id"].map(chain)

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 1.5]})
y_of = {cid: len(ex) - i for i, cid in enumerate(ex["id"])}
for ax in (ax_l, ax_r):
    ax.set_ylim(0.3, len(ex) + 0.9)
    ax.axis("off")

# Sinistra: come lo registra il campo depth (1 o 2)
ax_l.set_xlim(0, 3)
for row in ex.itertuples():
    depth = 1 if pd.isna(row.parent) else 2
    x = 0.3 + 0.6 * (depth - 1)
    label = f"{row.id}  {row.author}: comment" if pd.isna(row.parent) else f"{row.id}  {row.author} replies to {row.parent_author}"
    ax_l.text(x, y_of[row.id], label, va="center", fontsize=9)
    ax_l.text(2.75, y_of[row.id], f"depth {depth}", va="center", ha="right", fontsize=8, color="#555555")
ax_l.set_title("NYT field depth: every reply at level 2", loc="left", fontsize=10)

# Destra: struttura ricostruita da parent_id, colonna = chain_depth
ax_r.set_xlim(0.4, 5.4)
for d in range(1, 5):
    ax_r.text(d, len(ex) + 0.6, f"chain_depth {d}", ha="center", fontsize=8, color="#555555")
pos = {row.id: (row.chain_depth, y_of[row.id]) for row in ex.itertuples()}
for row in ex.itertuples():
    if not pd.isna(row.parent):
        ax_r.annotate("", xy=(pos[row.parent][0] + 0.2, pos[row.parent][1]), xytext=(pos[row.id][0] - 0.2, pos[row.id][1]),
                      arrowprops={"arrowstyle": "->", "color": "#777777", "linewidth": 1})
for row in ex.itertuples():
    x, y = pos[row.id]
    deep = row.chain_depth >= 3  # stessa soglia di DEEP_MIN_DEPTH, definita nella cella successiva
    ax_r.scatter(x, y, s=650, zorder=3, facecolor=PERIOD_COLORS[1] if deep else "#e8f0fb",
                 edgecolor=MAIN_COLOR, linewidth=1)
    ax_r.text(x, y, row.author, zorder=4, ha="center", va="center", fontsize=9, fontweight="bold")
    ax_r.text(x + 0.28, y, row.id, ha="left", va="center", fontsize=8, color="#555555")
ax_r.text(4.55, 1.6, "orange fill: deep\n(chain_depth >= 3)\n\narrow: reply -> parent_id",
          fontsize=8, va="center", color="#333333")
ax_r.set_title("Rebuilt from parent_id", loc="left", fontsize=10)
fig.suptitle("Thread structure: depth field vs reconstructed reply chains (§4)", x=0.01, ha="left", fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "comments_thread_structure.png", dpi=200, bbox_inches="tight")
plt.show()


**Come leggere lo schema.** Un thread inventato con tre lettori (A, B, C), visto in due modi. A sinistra il campo `depth`: tutte le risposte stanno al livello 2. A destra la struttura ricostruita da `parent_id`, con una colonna per `chain_depth`: R2 risponde a R1 e R4 a R2, quindi sono risposte in catena (arancioni); R1 e R3 rispondono al commento iniziale. Lo schema è salvato in `docs/figures/comments_thread_structure.png`.

In [ ]:
# Prima di questa data il NYT registrava ogni risposta come rivolta al commento iniziale del thread
THREADED_REPLIES_START = pd.Timestamp("2018-04-01", tz="America/New_York")
DEEP_MIN_DEPTH = 3


def add_chain_depth(df: pd.DataFrame) -> pd.DataFrame:
    """chain_depth risalendo parent_id fino al commento iniziale. Risposte con padre non raccolto = 2."""
    depth = pd.Series(np.where(df["is_reply"], 2, 1), index=df["comment_id"].values)
    has_parent = df["parent_id"].notna() & df["parent_id"].isin(df["comment_id"])
    children = df.loc[has_parent, "comment_id"].values
    parents = df.loc[has_parent, "parent_id"].astype("int64").values
    # propagazione a punto fisso: ogni giro allunga di un livello le catene ancora incomplete
    for _ in range(100):
        new_depth = depth.copy()
        new_depth.loc[children] = depth.loc[parents].values + 1
        if new_depth.equals(depth):
            break
        depth = new_depth
    else:
        raise RuntimeError("chain_depth non converge: ciclo in parent_id?")
    df = df.copy()
    df["chain_depth"] = depth.loc[df["comment_id"]].values
    df["deep"] = df["chain_depth"] >= DEEP_MIN_DEPTH
    return df


comments = add_chain_depth(comments)
# Giorno in ora di New York: unita' di tempo della serie della catena e base delle settimane
comments["day"] = comments["created_at"].dt.tz_convert("America/New_York").dt.tz_localize(None).dt.normalize()
comments["threaded"] = comments["created_at"] >= THREADED_REPLIES_START

# Metriche per articolo, salvate per il confronto con il sentiment dei commenti (notebook 11)
thread_stats = comments.groupby("article_id").agg(
    commenti=("comment_id", "size"),
    risposte=("is_reply", "sum"),
    profondita_max=("chain_depth", "max"),
    risposte_in_catena=("deep", "sum"),
    threaded=("threaded", "all"),
)
thread_stats["profondita_media_risposte"] = comments[comments["is_reply"]].groupby("article_id")["chain_depth"].mean().round(2)
thread_stats["quota_in_catena"] = (thread_stats["risposte_in_catena"] / thread_stats["commenti"]).round(3)
thread_stats = thread_stats.reset_index().merge(
    article_stats[["article_id", "article_pub_date", "period", "section_name", "headline"]], on="article_id", how="left"
).sort_values("article_pub_date")
thread_stats.to_csv(OUT_THREAD_METRICS, index=False)
log.info("Scritto %s (%d articoli)", OUT_THREAD_METRICS.relative_to(BASE_DIR), len(thread_stats))

# Quota di risposte in catena per periodo, solo nell'era con catene registrate
deep_share = comments[comments["threaded"]].groupby("period", observed=True)["deep"].mean().mul(100).round(1)
log.info("Risposte in catena dal 2018-04, %% dei commenti: %s; catena più lunga: %d livelli",
         deep_share.to_dict(), comments["chain_depth"].max())


**Come leggere l'output.** Su tutto il corpus 29.572 risposte sono in catena (21,2% delle risposte) e la catena più lunga arriva a 12 livelli. Dall'aprile 2018 la quota di risposte in catena è stabile tra i periodi (11,2-12,7%, nel log): in media il 2025-presente non discute più a lungo degli altri, quindi le differenze vanno cercate nel tempo, con i burst. Le metriche per articolo sono in `comments_thread_metrics.csv`, per il confronto con il sentiment (notebook 11).

### 4.1 Algoritmo di Kleinberg

Il metodo di Kleinberg (2002), *Bursty and Hierarchical Structure in Streams*, decide quando un aumento è abbastanza forte e lungo da non essere un'oscillazione normale. La serie è trattata come prodotta da un automa con stati nascosti: lo stato 0 emette al ritmo di fondo, lo stato *i* a un ritmo *s^i* volte più alto. Salire di un livello costa `γ · ln(T)` (T = numero di unità di tempo della serie), scendere è gratis. L'algoritmo di Viterbi trova la sequenza di stati a costo minimo. Un **burst** è un tratto consecutivo con stato ≥ 1 e il suo **livello** è lo stato massimo raggiunto: con s = 2 i livelli 1, 2 e 3 corrispondono a circa 2, 4 e 8 volte il fondo. Parametri di riferimento: s = 2, γ = 1, 3 livelli, i valori proposti da Kleinberg.

| Serie | Emissione | Unità | Ritmo di fondo |
|---|---|---|---|
| Volume | Poisson sui conteggi | settimana, settimane vuote incluse | media settimanale del periodo |
| Catena | binomiale sulle quote (versione *batched* di Kleinberg) | giorno con commenti | quota del periodo |

**Scelte.**

- **Settimana per il volume.** Con il giorno ogni articolo molto commentato diventa un burst di 1-2 giorni, e il risultato è l'elenco degli articoli più commentati (verifica nel §4.3). Con la settimana emergono fasi di più settimane.
- **Giorno per la catena.** La misura è una quota: un giorno con pochi commenti pesa poco nella verosimiglianza binomiale e non frammenta la serie. Con la settimana un giorno di discussione intensa si diluirebbe negli altri sei.
- **Fondo per periodo.** Sul corpus intero il fondo sarebbe dominato dal 2025-presente (196 mila commenti su 286 mila): quel periodo diventerebbe un unico burst e prima non emergerebbe quasi nulla. Ogni burst è quindi eccezionale rispetto al suo periodo, e i livelli non si confrontano tra periodi.
- **Taglio sui buchi** (`MAX_GAP_DAYS = 3`). La serie della catena contiene solo i giorni con commenti, quindi due giorni lontani mesi sarebbero consecutivi per l'automa. Un burst si chiude quando tra due giorni della serie passano più di 3 giorni (un thread si chiude quasi sempre entro 24-72 ore).

**Limiti.** Nei periodi con pochi articoli commentati (2003-2016: 72; 2021-2024: 32) il fondo settimanale è vicino a zero e quasi ogni settimana con commenti diventa un burst di volume. La Poisson assume commenti indipendenti, mentre arrivano a grappoli. Le date fino al 2018 sono in parte imputate (§1), con uno scarto irrilevante a scala di giorno o settimana.

In [ ]:
# Stati 0-3; s e gamma sono i valori di riferimento proposti da Kleinberg (sensibilita' in §4.3)
N_LEVELS = 3
KLEINBERG_S = 2.0
KLEINBERG_GAMMA = 1.0


def viterbi_states(costs: np.ndarray, gamma: float) -> np.ndarray:
    """Sequenza di stati a costo minimo. costs: (T, K) costo di emissione; salire costa gamma*ln(T) per livello."""
    n_units, n_states = costs.shape
    up_cost = gamma * np.log(n_units)
    trans = np.array([[max(j - i, 0) * up_cost for j in range(n_states)] for i in range(n_states)])
    total = costs[0] + trans[0]
    back = np.zeros((n_units, n_states), dtype=int)
    for t in range(1, n_units):
        candidates = total[:, None] + trans
        back[t] = candidates.argmin(axis=0)
        total = candidates.min(axis=0) + costs[t]
    states = np.zeros(n_units, dtype=int)
    states[-1] = total.argmin()
    for t in range(n_units - 1, 0, -1):
        states[t - 1] = back[t, states[t]]
    return states


def kleinberg_counts(counts, s: float = KLEINBERG_S, gamma: float = KLEINBERG_GAMMA, n_levels: int = N_LEVELS) -> np.ndarray:
    """Variante per conteggi: emissione Poisson con ritmo base * s^i (costo = -log verosimiglianza senza costanti)."""
    counts = np.asarray(counts, dtype=float)
    # Ritmo di fondo = media della serie, cioe' del periodo
    rates = counts.mean() * s ** np.arange(n_levels + 1)
    costs = rates[None, :] - counts[:, None] * np.log(rates)[None, :]
    return viterbi_states(costs, gamma)


def kleinberg_proportions(relevant, totals, s: float = KLEINBERG_S, gamma: float = KLEINBERG_GAMMA, n_levels: int = N_LEVELS) -> np.ndarray:
    """Variante batched di Kleinberg: r_t eventi rilevanti su d_t, emissione binomiale con p0 * s^i (< 1)."""
    r = np.asarray(relevant, dtype=float)
    d = np.asarray(totals, dtype=float)
    p0 = r.sum() / d.sum()
    probs = p0 * s ** np.arange(n_levels + 1)
    # Gli stati con probabilita' >= 1 non sono definiti e vengono scartati
    probs = probs[probs < 1]
    costs = -(r[:, None] * np.log(probs)[None, :] + (d - r)[:, None] * np.log1p(-probs)[None, :])
    return viterbi_states(costs, gamma)


# Oltre 3 giorni senza commenti un burst di conflitto si chiude (i thread durano 24-72 ore)
MAX_GAP_DAYS = 3


def state_runs(states: np.ndarray, index: pd.Index | None = None, max_gap_days: int = MAX_GAP_DAYS) -> list[tuple[int, int, int]]:
    """Tratti consecutivi con stato >= 1: (inizio, fine inclusa, livello massimo).

    Con `index` (date delle unita'), un tratto si spezza dove due unita' consecutive distano piu' di
    max_gap_days: nelle serie del conflitto i giorni senza commenti non ci sono, e senza questo taglio
    lo stato alto "sopravvive" per mesi tra due articoli lontani.
    """
    runs, start = [], None
    # nelle serie settimanali le unita' distano sempre 7 giorni: il taglio scatta solo oltre il passo della serie
    step_days = int(pd.Series(index).diff().min().days) if index is not None and len(index) > 1 else 1
    for t in range(len(states) + 1):
        state = states[t] if t < len(states) else 0
        gap = (
            index is not None and start is not None and t < len(states)
            and (index[t] - index[t - 1]).days > max(max_gap_days, step_days)
        )
        if start is not None and (state == 0 or gap):
            runs.append((start, t - 1, int(max(states[start:t]))))
            start = None
        if state > 0 and start is None:
            start = t
    return runs

### 4.2 Burst di volume

Per ogni periodo (articoli assegnati per anno di pubblicazione, come nel resto del progetto) si costruiscono le due serie e si applica Kleinberg:

- **volume**: commenti per settimana (lunedì-domenica, ora di New York), settimane vuote incluse: una settimana senza commenti è un dato, non un buco;
- **catena**: per ogni giorno con commenti dopo `THREADED_REPLIES_START`, risposte in catena sul totale dei commenti del giorno.

Per ogni burst si registrano date, livello, commenti, quota di risposte in catena nel burst e nel periodo, giorno di picco e i 3 articoli che contribuiscono di più (`comments_bursts.csv`). Il grafico mostra i burst di volume; quelli di catena sono confrontati con il volume nel §4.4.

In [ ]:
# unit: W = settimana, D = giorno; relevant: colonna booleana contata come evento (None = tutti i commenti)
SERIES = {
    "volume": {"unit": "W", "relevant": None},
    "catena": {"unit": "D", "relevant": "deep"},
}


def unit_index(days: pd.Series, unit: str) -> pd.Series:
    return days.dt.to_period("W-SUN").dt.start_time if unit == "W" else days


def build_series(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Serie (unita' di tempo -> totali, rilevanti) per un periodo, nel formato atteso da Kleinberg."""
    spec = SERIES[name]
    if spec["relevant"] is not None:
        df = df[df["threaded"]]
    if df.empty:
        return pd.DataFrame(columns=["totali", "rilevanti"])
    units = unit_index(df["day"], spec["unit"])
    grouped = df.groupby(units).agg(totali=("comment_id", "size"))
    grouped["rilevanti"] = df.groupby(units)[spec["relevant"]].sum() if spec["relevant"] else grouped["totali"]
    if spec["unit"] == "W":
        # settimane vuote incluse: per il volume un'assenza di commenti e' un dato, non un buco
        full = pd.date_range(grouped.index.min(), grouped.index.max(), freq="7D")
        grouped = grouped.reindex(full, fill_value=0)
    return grouped


def detect(series: pd.DataFrame, name: str, s: float = KLEINBERG_S, gamma: float = KLEINBERG_GAMMA) -> np.ndarray:
    if len(series) < 2 or series["rilevanti"].sum() == 0:
        return np.zeros(len(series), dtype=int)
    if SERIES[name]["relevant"] is None:
        return kleinberg_counts(series["totali"].values, s=s, gamma=gamma)
    return kleinberg_proportions(series["rilevanti"].values, series["totali"].values, s=s, gamma=gamma)


def describe_burst(df_period: pd.DataFrame, series: pd.DataFrame, name: str, start: int, end: int, level: int) -> dict:
    spec = SERIES[name]
    first = series.index[start]
    last = series.index[end] + (pd.Timedelta(days=6) if spec["unit"] == "W" else pd.Timedelta(0))
    window = df_period[(df_period["day"] >= first) & (df_period["day"] <= last)]
    if spec["relevant"]:
        window = window[window["threaded"]]
    rank_col = spec["relevant"] or "comment_id"
    by_article = window.groupby("article_id")[rank_col].agg("sum" if spec["relevant"] else "size").sort_values(ascending=False)
    top = by_article.head(3)
    titles = article_stats.set_index("article_id").loc[top.index]
    in_window = series.iloc[start:end + 1]
    row = {
        "serie": name,
        "inizio": first.date(),
        "fine": last.date(),
        "unita": end - start + 1,
        "livello": level,
        "commenti": len(window),
        "giorno_di_picco": window.groupby("day").size().idxmax().date() if len(window) else None,
        "articolo_principale": top.index[0] if len(top) else None,
        "articoli_principali": " | ".join(
            f"{t.article_pub_date.date()} {t.headline} ({n})" for (_, t), n in zip(titles.iterrows(), top.values)
        ),
    }
    if spec["relevant"]:
        row["quota_nel_burst"] = round(in_window["rilevanti"].sum() / in_window["totali"].sum(), 3)
        row["quota_periodo"] = round(series["rilevanti"].sum() / series["totali"].sum(), 3)
    return row


# Kleinberg separato per ogni (periodo, serie): il fondo e' sempre quello del periodo
period_labels = [label for label, _, _ in get_periods(int(comments["year"].max()))]
series_store, burst_rows = {}, []
for period in period_labels:
    df_period = comments[comments["period"] == period]
    for name in SERIES:
        series = build_series(df_period, name)
        states = detect(series, name)
        series_store[(period, name)] = (series, states)
        for start, end, level in state_runs(states, series.index):
            burst_rows.append({"periodo": period, **describe_burst(df_period, series, name, start, end, level)})

bursts = pd.DataFrame(burst_rows)


In [ ]:
# Eventi dei burst di volume più forti, letti dagli articoli trainanti (colonna articoli_principali di bursts)
VOLUME_EVENTS = {
    "2017-02-20": "new deportation rules",
    "2018-06-11": "family separation",
    "2019-04-01": "Nielsen resigns",
    "2019-07-08": "announced raids",
    "2024-11-18": "military for deportations",
    "2025-06-09": "'This Is What Autocracy\nLooks Like' (editorial)",
    "2026-02-02": "Renee Good killing,\nprotests",
}

fig, axes = plt.subplots(len(period_labels), 1, figsize=(11, 2.4 * len(period_labels)))
for ax, period in zip(axes, period_labels):
    series, states = series_store[(period, "volume")]
    in_burst = states > 0
    colors = [PERIOD_COLORS[1] if b else MUTED_COLOR for b in in_burst]
    ax.bar(series.index, series["totali"] / 1000, width=pd.Timedelta(days=6), color=colors, align="edge")
    share = series.loc[in_burst, "totali"].sum() / series["totali"].sum()
    n_bursts = len(state_runs(states, series.index))
    ax.set_title(f"{period}: {n_bursts} bursts, {share:.0%} of the period's comments in burst weeks", loc="left", fontsize=10)
    ax.set_ylabel("comments / week\n(thousands)", fontsize=8)
    ax.set_ylim(0, series["totali"].max() / 1000 * 1.35)
    period_bursts = bursts[(bursts["serie"] == "volume") & (bursts["periodo"] == period)]
    for row in period_bursts.itertuples():
        label = VOLUME_EVENTS.get(str(row.inizio))
        if label is None:
            continue
        week = pd.Timestamp(row.giorno_di_picco).to_period("W-SUN").start_time
        peak = series["totali"].get(week, series["totali"].max()) / 1000
        pos = (week - series.index.min()) / (series.index.max() - series.index.min())
        ha = "left" if pos < 0.12 else "right" if pos > 0.88 else "center"
        ax.annotate(label, xy=(week, peak), xytext=(0, 6), textcoords="offset points",
                    ha=ha, va="bottom", fontsize=7.5, color="#333")
handles = [plt.Rectangle((0, 0), 1, 1, color=PERIOD_COLORS[1]), plt.Rectangle((0, 0), 1, 1, color=MUTED_COLOR)]
axes[0].legend(handles, ["week in a burst", "normal week"], frameon=False, fontsize=8, loc="upper right")
fig.suptitle("Volume bursts: weeks with exceptionally many comments, per period (Kleinberg, s = 2, γ = 1)",
             x=0.01, ha="left", fontsize=11)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_bursts_volume.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "comments_bursts_volume.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

**Come leggere il grafico.** Una riga per periodo, una barra per settimana: arancione se la settimana è in un burst di volume, grigia altrimenti. Nel titolo di ogni riga il numero di burst e la quota dei commenti del periodo che cade in settimane di burst. Le etichette indicano gli eventi dei burst più forti, letti dagli articoli che vi raccolgono più commenti. La scala verticale cambia da una riga all'altra, perché ogni periodo ha il suo fondo.

- **2017-2020 e 2025-presente: i burst sono eventi.** Le nuove regole di espulsione (febbraio 2017), la separazione delle famiglie (giugno 2018), le dimissioni di Kirstjen Nielsen (aprile 2019), i raid annunciati (luglio 2019); nel 2025-presente l'editoriale "This Is What Autocracy Looks Like" (giugno 2025) e l'uccisione di Renee Good con le proteste che ne seguono (febbraio 2026), entrambi intorno ai 15.000 commenti a settimana.
- **Nel 2025-presente la discussione è continua.** Quasi ogni settimana ha commenti, e i 13 burst raccolgono il 47% dei commenti: emergono solo le settimane davvero eccezionali.
- **2003-2016 e 2021-2024: il volume dice poco.** Con pochi articoli commentati quasi ogni settimana con commenti è un burst (96% e 97% dei commenti in burst): i burst coincidono con i singoli articoli (limiti nel §4.1).

### 4.3 Sensibilità ai parametri

I burst dipendono da `s` e `γ`. Griglia `s ∈ {1.5, 2, 3}` × `γ ∈ {0.5, 1, 2}` (9 configurazioni): per ognuna si conta quanto le unità di tempo in burst coincidono con quelle del riferimento (Jaccard). Per ogni burst del riferimento, `stabilita` = quota delle 9 configurazioni in cui almeno metà delle sue unità resta in un burst; un burst è **stabile** se resta in almeno 7 configurazioni su 9. Solo i burst stabili si usano per le conclusioni. Il volume è ricalcolato anche a giorno, per verificare la scelta della settimana.

In [ ]:
# Griglia 3 x 3 attorno al riferimento (s = 2, gamma = 1); volume_giorno serve solo a verificare la scelta della settimana
S_GRID = [1.5, 2.0, 3.0]
GAMMA_GRID = [0.5, 1.0, 2.0]


def burst_units(states: np.ndarray, index: pd.Index) -> set:
    return set(index[states > 0])


sensitivity_rows = []
per_config_units = {}
daily_volume_lengths = []
for name in list(SERIES) + ["volume_giorno"]:
    for s in S_GRID:
        for gamma in GAMMA_GRID:
            units, n_bursts = set(), 0
            for period in period_labels:
                if name == "volume_giorno":
                    df_period = comments[comments["period"] == period]
                    counts = df_period.groupby("day").size()
                    counts = counts.reindex(pd.date_range(counts.index.min(), counts.index.max(), freq="D"), fill_value=0)
                    states = kleinberg_counts(counts.values, s=s, gamma=gamma)
                    index = counts.index
                else:
                    series, _ = series_store[(period, name)]
                    states = detect(series, name, s=s, gamma=gamma)
                    index = series.index
                units |= {(period, u) for u in burst_units(states, index)}
                n_bursts += len(state_runs(states, index))
                if (name, s, gamma) == ("volume_giorno", KLEINBERG_S, KLEINBERG_GAMMA):
                    daily_volume_lengths += [end - start + 1 for start, end, _ in state_runs(states, index)]
            per_config_units[(name, s, gamma)] = units
            sensitivity_rows.append({"serie": name, "s": s, "gamma": gamma, "burst": n_bursts})

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity["jaccard_vs_riferimento"] = [
    len(per_config_units[(r.serie, r.s, r.gamma)] & per_config_units[(r.serie, KLEINBERG_S, KLEINBERG_GAMMA)])
    / max(1, len(per_config_units[(r.serie, r.s, r.gamma)] | per_config_units[(r.serie, KLEINBERG_S, KLEINBERG_GAMMA)]))
    for r in sensitivity.itertuples()
]
lengths = pd.Series(daily_volume_lengths)
print(f"Volume a giorno (riferimento): {len(lengths)} burst, durata mediana {lengths.median():.0f} giorni, "
      f"{(lengths <= 2).mean():.0%} di 1-2 giorni")

In [ ]:
# Stabilita' = quota delle 9 configurazioni in cui almeno meta' delle unita' del burst resta in un burst
def burst_stability(row) -> float:
    series, _ = series_store[(row.periodo, row.serie)]
    start = pd.Timestamp(row.inizio)
    if SERIES[row.serie]["unit"] == "W":
        start = start.to_period("W-SUN").start_time
    units = {(row.periodo, u) for u in series.index if start <= u <= pd.Timestamp(row.fine)}
    kept = [
        len(units & per_config_units[(row.serie, s, gamma)]) >= len(units) / 2
        for s in S_GRID for gamma in GAMMA_GRID
    ]
    return round(float(np.mean(kept)), 2)


bursts["stabilita"] = [burst_stability(row) for row in bursts.itertuples()]


In [ ]:
# Accordo con il riferimento (Jaccard sulle unità in burst) per ogni (s, gamma); in titolo i burst stabili
SERIES_LABELS = {"volume": "volume (weeks)", "catena": "reply chains (days)"}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
fig.subplots_adjust(wspace=0.45)
for ax, name in zip(axes, SERIES):
    grid = sensitivity[sensitivity["serie"] == name].pivot(index="gamma", columns="s", values="jaccard_vs_riferimento")
    im = ax.imshow(grid.values, cmap="Blues", vmin=0, vmax=1, origin="lower")
    for (r, c), v in np.ndenumerate(grid.values):
        ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=9, color="white" if v > 0.6 else "#333")
    ax.set_xticks(range(len(S_GRID)), [f"s = {s:g}" for s in grid.columns])
    ax.set_yticks(range(len(GAMMA_GRID)), [f"γ = {g:g}" for g in grid.index])
    ax.grid(False)
    stability = bursts.loc[bursts["serie"] == name, "stabilita"]
    ax.set_title(f"{SERIES_LABELS[name]}\n{(stability >= 7 / 9).sum()}/{len(stability)} bursts stable", loc="left", fontsize=10)
fig.colorbar(im, ax=axes, label="agreement with reference (Jaccard)", shrink=0.85)
fig.suptitle("Sensitivity of bursts to s and γ (reference: s = 2, γ = 1)", x=0.01, ha="left", fontsize=11, y=1.04)
fig.savefig(PLOTS_DIR / "comments_bursts_sensitivity.png", dpi=200, bbox_inches="tight")
plt.show()

**Come leggere il grafico.** Per ciascuna serie, una cella per combinazione di `s` (colonne) e `γ` (righe): accordo con il riferimento `s = 2`, `γ = 1` (cella centrale, 1,00), misurato come Jaccard sulle unità di tempo in burst (1 = stesse unità, 0 = nessuna in comune). Nel titolo i burst stabili, presenti in almeno 7 configurazioni su 9.

- **Volume robusto**: accordo tra 0,87 e 0,99 ovunque, 85 burst stabili su 96.
- **Catena sensibile a `s`**: 0,71-0,80 se cambia solo `γ` (colonna centrale), 0,27-0,51 se cambia `s`. Solo 16 burst su 34 sono stabili: nel §4.4 i burst instabili sono disegnati vuoti e vanno citati con cautela.
- **Volume a giorno** (riga stampata): 239 burst, durata mediana 1 giorno. Sono i giorni dei singoli articoli molto commentati, il che conferma la scelta della settimana.

### 4.4 Catena contro volume

Un burst di catena che cade dentro un burst di volume può essere un effetto dell'affluenza: con più commenti crescono anche le occasioni di risposta. Un burst di catena **tocca il volume** se ha almeno un giorno dentro un burst di volume dello stesso periodo (un burst settimanale copre 7 giorni).

Come riferimento si usa la quota dei giorni della serie della catena coperti da burst di volume: se catena e volume fossero indipendenti, la quota di burst di catena che toccano il volume sarebbe di quest'ordine (un po' più alta, perché un burst di più giorni ha più occasioni di toccarne uno).

In [ ]:
# Giorni coperti dai burst di volume di un periodo: ogni settimana in burst vale 7 giorni
def volume_burst_days(period: str) -> set:
    series, states = series_store[(period, "volume")]
    days = set()
    for start, end, _ in state_runs(states, series.index):
        days |= set(pd.date_range(series.index[start], series.index[end] + pd.Timedelta(days=6), freq="D"))
    return days


volume_days = {p: volume_burst_days(p) for p in period_labels}
chain = bursts[bursts["serie"] == "catena"].copy()
chain["in_burst_volume"] = [
    bool(set(pd.date_range(row.inizio, row.fine, freq="D")) & volume_days[row.periodo]) for row in chain.itertuples()
]
bursts = bursts.merge(chain[["serie", "periodo", "inizio", "in_burst_volume"]], on=["serie", "periodo", "inizio"], how="left")
bursts.to_csv(OUT_BURSTS, index=False)
log.info("Scritto %s (%d burst)", OUT_BURSTS.relative_to(BASE_DIR), len(bursts))

chain["stabile"] = chain["stabilita"] >= 7 / 9
# Quanto la catena supera la quota del periodo: 2 = il doppio delle risposte in catena del solito
chain["intensita"] = chain["quota_nel_burst"] / chain["quota_periodo"]
chain_periods = [p for p in period_labels if not series_store[(p, "catena")][0].empty]

fig = plt.figure(figsize=(13, 2.5 * len(chain_periods)))
gs = fig.add_gridspec(len(chain_periods), 2, width_ratios=[3.2, 1], wspace=0.28, hspace=0.45)
inside_color, outside_color = MAIN_COLOR, PERIOD_COLORS[1]
for r, period in enumerate(chain_periods):
    ax = fig.add_subplot(gs[r, 0])
    series, _ = series_store[(period, "catena")]
    vol_series, vol_states = series_store[(period, "volume")]
    for start, end, _ in state_runs(vol_states, vol_series.index):
        ax.axvspan(vol_series.index[start], vol_series.index[end] + pd.Timedelta(days=7), color=MUTED_COLOR, alpha=0.45, linewidth=0)
    sub_chain = chain[chain["periodo"] == period]
    labelled_months = set()
    for row in sub_chain.itertuples():
        color = inside_color if row.in_burst_volume else outside_color
        ax.scatter(pd.Timestamp(row.inizio), row.intensita, s=25 + np.sqrt(row.commenti) * 3, zorder=3,
                   facecolor=color if row.stabile else "white", edgecolor=color, linewidth=1.4)
        month = pd.Timestamp(row.inizio).strftime("%b %Y")
        # una sola etichetta per mese: i burst di dicembre 2025 sono a pochi giorni l'uno dall'altro
        if row.stabile and not row.in_burst_volume and month not in labelled_months:
            labelled_months.add(month)
            ax.annotate(month, xy=(pd.Timestamp(row.inizio), row.intensita),
                        xytext=(0, 13), textcoords="offset points", ha="center", fontsize=7.5, color="#333")
    ax.set_xlim(series.index.min() - pd.Timedelta(days=10), series.index.max() + pd.Timedelta(days=10))
    ax.set_ylim(1, max(3.2, sub_chain["intensita"].max() * 1.25) if len(sub_chain) else 3)
    ax.set_ylabel("reply-chain share\n× period average", fontsize=8)
    ax.set_title(f"{period}" + (" (chains recorded from Apr 2018)" if period == "2017-2020" else ""), loc="left", fontsize=10)

# Destra: test di indipendenza, quota di burst di catena dentro il volume contro la quota attesa (giorni coperti)
ax = fig.add_subplot(gs[:, 1])
observed = chain.groupby("periodo")["in_burst_volume"].mean().reindex(chain_periods) * 100
expected = pd.Series({p: np.mean([d in volume_days[p] for d in series_store[(p, "catena")][0].index]) * 100 for p in chain_periods})
n_chain = chain.groupby("periodo").size().reindex(chain_periods)
y = np.arange(len(chain_periods))[::-1]
ax.barh(y + 0.18, observed.values, height=0.34, color=inside_color, label="chain bursts inside a volume burst")
ax.barh(y - 0.18, expected.values, height=0.34, color=MUTED_COLOR, label="expected if independent\n(days covered by volume bursts)")
for yy, o, e in zip(y, observed.values, expected.values):
    ax.text(o + 2, yy + 0.18, f"{o:.0f}%", va="center", fontsize=8)
    ax.text(e + 2, yy - 0.18, f"{e:.0f}%", va="center", fontsize=8)
ax.set_yticks(y, [f"{p}\n({n} chain bursts)" for p, n in zip(chain_periods, n_chain)], fontsize=8)
ax.set_xlim(0, 115)
ax.set_xlabel("%")
ax.legend(frameon=False, fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.06))
ax.set_title("Do chain bursts follow volume?", loc="left", fontsize=10)

handles = [
    plt.Rectangle((0, 0), 1, 1, color=MUTED_COLOR, alpha=0.45),
    plt.Line2D([], [], marker="o", ls="", markerfacecolor=outside_color, markeredgecolor=outside_color),
    plt.Line2D([], [], marker="o", ls="", markerfacecolor=inside_color, markeredgecolor=inside_color),
    plt.Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor="#555"),
]
fig.legend(handles, ["volume burst", "chain burst outside volume (stable ones dated)", "chain burst inside volume",
                     "hollow = not stable (< 7/9 settings)"], frameon=False, fontsize=8, ncol=4,
           loc="upper left", bbox_to_anchor=(0.01, 0.995))
fig.suptitle("Reply-chain bursts vs volume bursts", x=0.01, y=1.03, ha="left", fontsize=11)
fig.savefig(PLOTS_DIR / "comments_bursts_chain.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "comments_bursts_chain.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

**Come leggere il grafico.** A sinistra, una riga per periodo con catene registrate: le fasce grigie sono i burst di volume, i cerchi i burst di catena, posti alla data di inizio e all'altezza della loro intensità (quota di risposte in catena nel burst divisa per la quota media del periodo: 2 = il doppio del solito). Blu = il burst di catena tocca un burst di volume, arancione = cade fuori; pieno = stabile, vuoto = instabile; la dimensione cresce con i commenti. I burst stabili fuori dal volume hanno il mese in etichetta. A destra il test: quota di burst di catena che toccano un burst di volume (blu) contro la quota attesa se le due serie fossero indipendenti, cioè la quota di giorni coperti da burst di volume (grigio).

- **2017-2020 e 2021-2024: nessuna conclusione.** 12 burst di catena su 13 toccano un burst di volume, ma il volume copre già il 55% e il 69% dei giorni: la sovrapposizione è in gran parte attesa.
- **2025-presente: le discussioni lunghe non seguono l'affluenza.** Solo 6 burst di catena su 21 (29%) toccano un burst di volume, quasi esattamente la quota attesa per serie indipendenti (25%). I burst arancioni stabili cadono su editoriali e vicende locali: l'editoriale sui raid di Nashville (maggio 2025), le operazioni federali in Louisiana e a New Orleans e il processo a un giudice accusato di aver aiutato un immigrato a sfuggire agli agenti (dicembre 2025), i Dreamers espulsi nonostante il DACA (maggio 2026), un'operazione ICE in una cittadina (agosto 2026).

**Risultato del §4.** I burst di volume coincidono con eventi noti e sono robusti ai parametri. La quota media di risposte in catena è stabile tra i periodi, ma nel 2025-presente i giorni con discussioni insolitamente lunghe non coincidono con quelli di maggiore affluenza: emergono su editoriali, vicende locali e temi laterali, che l'analisi del solo volume non segnalerebbe. Prima del 2025 il confronto è poco informativo, perché i burst di volume coprono gran parte dei giorni. I dettagli di ogni burst (date, articoli trainanti, stabilità) sono in `comments_bursts.csv`.

## 5. Topic dei commenti (BERTopic)

**Obiettivo:** individuare di cosa parlano i lettori, con quali parole, e come cambia tra periodi. La pipeline è quella BERTopic del notebook 04 (articoli NYT), adattata a testi brevi e conversazionali e a un corpus cento volte più grande. Lo schema della pipeline completa è dopo il §5.6.

**Decisioni** (motivate nelle sottosezioni):

| Decisione | Scelta | Dove |
|---|---|---|
| Pulizia | menzioni `@utente`, nome del destinatario in apertura e URL rimossi prima di tutto | 6.1 |
| Soglia di lunghezza | **≥ 10 parole** sul testo pulito | 6.2 |
| Embedding | `all-mpnet-base-v2`, come notebook 04 e 07 (MiniLM fino al 2026-10-13, confronto in §5.10) | 6.3 |
| Addestramento | campione **bilanciato per periodo**, poi `transform` su tutti i commenti | 6.4 |
| Parametri | UMAP come notebook 04; `min_topic_size` adattato alla scala; c-TF-IDF + KeyBERTInspired | 6.5 |
| Outlier | **lasciati** in questa prima esecuzione, misurati per periodo e lunghezza | 6.6 |
| Stabilità | controllo rapido su 4 seed (AMI) | 6.8 |

### 5.1 Pulizia del testo

Le risposte si aprono quasi sempre con il nome della persona a cui si risponde (`@Margot. So true.`). 
Quindi: (1) ogni `@` seguito dal nome visualizzato di un utente noto viene rimosso fino alla fine del nome; (2) nelle risposte si rimuove anche il nome dell'utente a cui si risponde quando apre il testo senza `@` (`Rhys Yes. This is…`); (3) URL rimossi; (4) spazi normalizzati.

`body_clean` e `word_count_clean` sono aggiunti a `comments`.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MAX_NAME_TOKENS = 5


def load_display_names() -> tuple[set, pd.Series]:
    """Nomi visualizzati (solo in memoria, per la pulizia) e nome dell'utente a cui risponde ogni commento."""
    raw = pd.read_parquet(IN_COMMENTS, columns=["comment_id", "user_display_name", "parent_user_display_name"])
    names = set(raw["user_display_name"].dropna()) | set(raw["parent_user_display_name"].dropna())
    return names, raw.set_index("comment_id")["parent_user_display_name"]


def strip_mentions(text: str, names: set) -> str:
    out, i = [], 0
    while (j := text.find("@", i)) >= 0:
        out.append(text[i:j])
        rest = text[j + 1:]
        tokens = re.split(r"(\s+)", rest)  # separatori conservati per ricostruire la lunghezza esatta
        cut = None
        for k in range(min(MAX_NAME_TOKENS, (len(tokens) + 1) // 2), 0, -1):
            candidate = "".join(tokens[:2 * k - 1])
            if candidate.rstrip(":,;.!?") in names:
                cut = len(candidate)
                break
        if cut is None:
            match = re.match(r"[\w.\-]+", rest)
            cut = match.end() if match else 0
        out.append(" ")
        i = j + 1 + cut
    out.append(text[i:])
    return "".join(out)


def clean_body(body: str, parent_name: str | None, names: set) -> str:
    text = strip_mentions(body, names) if "@" in body else body
    text = re.sub(r"\s+", " ", URL_RE.sub(" ", text)).strip()
    if isinstance(parent_name, str) and parent_name and text.startswith(parent_name):
        text = text[len(parent_name):]
    return text.strip(" :,;-–—")


# I nomi visualizzati restano solo in memoria e vengono cancellati subito dopo la pulizia
display_names, parent_names = load_display_names()
comments["body_clean"] = [
    clean_body(body or "", parent_names.get(cid), display_names)
    for cid, body in zip(comments["comment_id"], comments["body"])
]
comments["word_count_clean"] = comments["body_clean"].str.split().str.len().fillna(0).astype(int)
del display_names, parent_names

changed = comments["body_clean"] != comments["body"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
log.info("Testi modificati dalla pulizia (menzioni, nomi, URL): %d (%.1f%%)", changed.sum(), changed.mean() * 100)
comments.loc[changed, ["body", "body_clean"]].sample(6, random_state=1).map(lambda s: s[:110])

La pulizia modifica 118.964 testi, il 41,5% dei commenti. Successivamente verrà usata solamente la colonna body_clean.

### 5.2 Lunghezza dei commenti e soglia

Un commento troppo breve non ha un tema riconoscibile ("So true.", "Horrific and outrageous.", "One can hope."): l'embedding lo colloca vicino ad altri commenti brevi per tono, non per argomento, e il topic che ne esce raccoglie esclamazioni invece di temi. Serve quindi una soglia minima di parole, scelta guardando i dati.

La cella sotto mostra la distribuzione delle parole dopo la pulizia, la quota di commenti esclusa a diverse soglie (in totale, per tipo e per periodo), un istogramma con le due soglie candidate (10 e 12 parole) e sei commenti a caso per fascia di lunghezza.

In [ ]:
wc = comments["word_count_clean"]
display(wc.describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(0).to_frame("parole (testo pulito)").T)

# Quota esclusa per ogni soglia candidata: totale, commenti vs risposte, per periodo
threshold_rows = []
for t in [5, 8, 10, 12, 15, 20]:
    below = wc < t
    threshold_rows.append({
        "soglia": t,
        "esclusi_pct": round(below.mean() * 100, 1),
        "esclusi_commenti_pct": round(below[~comments["is_reply"]].mean() * 100, 1),
        "esclusi_risposte_pct": round(below[comments["is_reply"]].mean() * 100, 1),
        **{f"esclusi_{p}_pct": round(below[comments["period"] == p].mean() * 100, 1) for p in period_labels},
    })
display(pd.DataFrame(threshold_rows).set_index("soglia"))

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(wc.clip(upper=300), bins=np.arange(0, 302, 2), color=MAIN_COLOR)
for t, style in [(10, "-"), (12, ":")]:
    ax.axvline(t, color=PERIOD_COLORS[1], linewidth=1.2, linestyle=style, label=f"{t} parole")
ax.set_xlabel("parole per commento (testo pulito; > 300 accorpati a 300)")
ax.set_ylabel("commenti")
ax.set_title("Lunghezza dei commenti", loc="left")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_length_hist.png", dpi=200, bbox_inches="tight")
plt.show()

# Campioni per fascia di lunghezza: servono a vedere dove cominciano i commenti con un tema
for lo, hi in [(5, 8), (8, 10), (10, 12), (12, 15)]:
    sample = comments.loc[(wc >= lo) & (wc < hi), "body_clean"].sample(6, random_state=3)
    print(f"\n--- {lo}-{hi - 1} parole")
    for text in sample:
        print("  ", text[:130])

**Scelta: `MIN_WORDS = 10`** (nell'intervallo 10-12 deciso a monte). Esclude il 6,1% dei commenti: il 3,7% dei commenti all'articolo e l'8,7% delle risposte, che sono più brevi. La quota esclusa è simile in tutti i periodi (dal 4,1% al 6,7%), quindi la soglia non penalizza un periodo più degli altri. Nei campioni sotto le 8 parole prevalgono le reazioni ("Hurting people is the point.", "Thank you for your service and advice."); tra 8 e 11 parole i commenti sono misti, alcuni già con un tema riconoscibile ("Canada has strict immigration laws.", "Funding for ICE but nothing else.").

**Verifica a posteriori** (§5.6): se la fascia 10-11 parole finisse quasi tutta tra gli outlier, la soglia andrebbe alzata a 12. Risultato: outlier al 61,6% nella fascia 10-11 contro il 60,2% nella 12-14 e il 52-57% oltre.

In [ ]:
# Soglia scelta sopra; i commenti sotto soglia restano nei dati ma non ricevono un topic
MIN_WORDS = 10
comments["topic_eligible"] = comments["word_count_clean"] >= MIN_WORDS
log.info("Commenti con almeno %d parole: %d su %d (%.1f%%)",
         MIN_WORDS, comments["topic_eligible"].sum(), len(comments), comments["topic_eligible"].mean() * 100)

### 5.3 Embedding

Un embedding trasforma ogni commento pulito in un vettore di 768 numeri, costruito in modo che testi con significato simile abbiano vettori vicini. BERTopic raggruppa poi i commenti in base a questi vettori (§5.5): commenti con vettori vicini parlano di cose simili.

**Modello: `all-mpnet-base-v2`.** È addestrato su un insieme di circa un miliardo di coppie di frasi, con una parte grande di testo conversazionale (Reddit, forum di domande e risposte), ed è quindi adatto a commenti brevi e informali; è più grande e più accurato nei benchmark di sentence-transformers. È lo stesso modello dei notebook 04 (articoli NYT) e 07 (Guardian), e questo rende i topic dei commenti confrontabili con quelli degli articoli.

**Costo e cache.** Gli embedding sono calcolati per tutti i commenti puliti, anche quelli sotto soglia (costano poco), così cambiare `MIN_WORDS` non richiede di ricalcolarli. I vettori sono salvati a blocchi di 20.000 in `data/processed/nyt/comments_embeddings_mpnet_v1/` (embedding e `comment_id` per blocco): se il calcolo si interrompe, riparte dal primo blocco mancante.

In [ ]:
from sentence_transformers import SentenceTransformer

# il caricamento del modello fa decine di richieste HEAD a HuggingFace, ognuna loggata a INFO
for noisy in ("httpx", "huggingface_hub", "sentence_transformers"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

EMBEDDING_MODEL = "all-mpnet-base-v2"
# Da alzare se cambiano le regole di pulizia del §5.1: cambia la cartella e gli embedding si ricalcolano
CLEANING_VERSION = 1
EMBEDDING_CHUNK = 20_000
EMBEDDINGS_DIR = PROCESSED_DIR / f"comments_embeddings_mpnet_v{CLEANING_VERSION}"


def compute_embeddings(df: pd.DataFrame) -> pd.DataFrame:
    """Embedding di body_clean per tutti i commenti, a blocchi con checkpoint. Ritorna DataFrame indicizzato per comment_id."""
    EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
    ordered = df.sort_values("comment_id")[["comment_id", "body_clean"]].reset_index(drop=True)
    model = None
    for start in range(0, len(ordered), EMBEDDING_CHUNK):
        path = EMBEDDINGS_DIR / f"chunk_{start // EMBEDDING_CHUNK:03d}.npz"
        chunk = ordered.iloc[start:start + EMBEDDING_CHUNK]
        if path.exists() and np.array_equal(np.load(path)["comment_id"], chunk["comment_id"].values):
            continue
        model = model or SentenceTransformer(EMBEDDING_MODEL)
        log.info("Embedding blocco %d (%d commenti)...", start // EMBEDDING_CHUNK, len(chunk))
        vectors = model.encode(chunk["body_clean"].tolist(), batch_size=64, show_progress_bar=False)
        np.savez(path, comment_id=chunk["comment_id"].values, embedding=vectors.astype(np.float32))
    parts = [np.load(p) for p in sorted(EMBEDDINGS_DIR.glob("chunk_*.npz"))]
    ids = np.concatenate([p["comment_id"] for p in parts])
    vectors = np.concatenate([p["embedding"] for p in parts])
    return pd.DataFrame(vectors, index=pd.Index(ids, name="comment_id"))


all_embeddings = compute_embeddings(comments)
log.info("Embedding disponibili: %d x %d", *all_embeddings.shape)

### 5.4 Campione bilanciato per periodo

Il 68,6% dei commenti sopra soglia è del 2025-presente. Addestrando BERTopic su tutti i commenti, i temi dei periodi precedenti finirebbero assorbiti nei temi del 2025-2026 o tra gli outlier: HDBSCAN (§5.5) forma un cluster solo dove i punti sono densi, e un tema discusso soprattutto nel 2017-2020 avrebbe pochi punti rispetto alla massa del 2025-presente. Il modello si addestra quindi su un campione con al massimo `N_PER_PERIOD = 15.000` commenti per periodo (tutti, se il periodo ne ha meno), estratti a caso con seed fisso. Dopo l'addestramento, `transform` assegna un topic a tutti i 268.866 commenti sopra soglia.

Il bilanciamento ha tre conseguenze da tenere presenti.

1. **Le dimensioni dei topic nel campione non rappresentano il corpus.** Il campione è deformato di proposito: il 2003-2016 pesa il 19,7% del campione contro il 4,0% dei commenti sopra soglia, il 2025-presente il 27,4% contro il 68,6%. I conteggi restituiti da BERTopic dopo l'addestramento (colonna `commenti_campione` del §5.7) non dicono quanto un tema sia diffuso tra i lettori.
2. **Le quote per periodo si leggono dopo `transform`.** Solo l'assegnazione a tutti i commenti rispecchia il corpus: la colonna `commenti_tutti` e le quote per periodo del §5.12 usano questa.
3. **Due periodi restano sotto la quota.** Il 2003-2016 ha 10.768 commenti sopra soglia e il 2021-2024 13.907: entrano tutti, ma pesano il 19,7% e il 25,4% del campione, contro il 27,4% degli altri due. Il 2003-2016 resta il periodo più debole nell'addestramento: un suo tema ha meno punti per formare un cluster denso e più probabilità di finire tra gli outlier.

In [ ]:
# Al massimo 15.000 commenti per periodo, estratti con seed fisso; i periodi piu' piccoli entrano interi
N_PER_PERIOD = 15_000
SAMPLE_SEED = 42

eligible = comments[comments["topic_eligible"]].reset_index(drop=True)
train = pd.concat(
    [g.sample(min(len(g), N_PER_PERIOD), random_state=SAMPLE_SEED) for _, g in eligible.groupby("period", observed=True)],
    ignore_index=True,
)
display(pd.DataFrame({
    "sopra_soglia": eligible["period"].value_counts(),
    "campione": train["period"].value_counts(),
}).reindex(period_labels))

train_docs = train["body_clean"].tolist()
train_embeddings = all_embeddings.loc[train["comment_id"]].values
eligible_docs = eligible["body_clean"].tolist()
eligible_embeddings = all_embeddings.loc[eligible["comment_id"]].values

Per ogni periodo, i commenti sopra soglia e quelli che entrano nel campione di addestramento: 54.675 in tutto. Il 2003-2016 e il 2021-2024 entrano interi, gli altri due periodi sono campionati a 15.000.

### 5.5 Modello

**Cosa fa il modello**, in quattro passaggi:

1. **UMAP** riduce gli embedding da 768 a 5 dimensioni conservando le vicinanze locali. Parametri del notebook 04: `n_neighbors=15`, `n_components=5`, `min_dist=0`, coseno, `random_state=42`.
2. **HDBSCAN** cerca le zone dense nello spazio ridotto: ogni zona densa diventa un topic, i punti che non cadono in nessuna zona densa sono outlier (topic −1).
3. **c-TF-IDF** sceglie le parole che descrivono ogni topic: frequenti nei commenti del topic e rare negli altri (default di BERTopic, come nel notebook 04).
4. **KeyBERTInspired** riordina le parole candidate per vicinanza semantica al centro del topic; su testi brevi e rumorosi dà etichette più leggibili. Sono salvate entrambe le rappresentazioni.

**Parametri.**
- `min_topic_size = 100`: il valore del notebook 04 (15) era tarato su 2.093 articoli; su circa 55.000 commenti produrrebbe centinaia di micro-topic (un thread che ripete la stessa battuta diventerebbe un topic). 100 è circa lo 0,2% del campione.
- **Stopword**: inglesi standard + "ice" (come nel notebook 04: presente ovunque, renderebbe le etichette tutte uguali) + frammenti di contrazioni spezzati dal tokenizer ("don", "doesn", "ll", "ve"…). `min_df = 5`, unigrammi e bigrammi.

**Nota sugli outlier** (per la seconda esecuzione). BERTopic costruisce HDBSCAN con `min_cluster_size = min_topic_size = 100` e non imposta `min_samples`, che quindi prende lo stesso valore. Un punto entra in un cluster solo se è denso rispetto ai suoi 100 vicini più prossimi, una condizione severa che contribuisce probabilmente alla quota alta di outlier (§5.6). Due strade per ridurli:

- passare a BERTopic un HDBSCAN esplicito con `min_samples` più basso (10-25): gli outlier calano, ma cambiano anche i topic, che andrebbero riletti e ricontrollati per stabilità;
- tenere il modello attuale e riassegnare gli outlier con `reduce_outliers`, come nel notebook 04: i topic restano gli stessi e si aggiungono assegnazioni più deboli, da marcare con un flag.

In [ ]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from umap import UMAP

# Circa lo 0,2% del campione: evita micro-topic fatti da un solo thread
MIN_TOPIC_SIZE = 100
TOPIC_SEED = 42
# "ice" escluso come nel notebook 04; frammenti di contrazioni lasciati dal tokenizer
COMMENT_STOPWORDS = list(ENGLISH_STOP_WORDS | {
    "ice", "don", "doesn", "didn", "isn", "aren", "wasn", "weren", "won", "wouldn", "shouldn", "couldn",
    "ll", "ve", "re", "just",
})
logging.getLogger("BERTopic").setLevel(logging.WARNING)
embedding_model = SentenceTransformer(EMBEDDING_MODEL)


def make_topic_model(seed: int, with_keybert: bool = True) -> BERTopic:
    return BERTopic(
        embedding_model=embedding_model,
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=seed),
        vectorizer_model=CountVectorizer(stop_words=COMMENT_STOPWORDS, ngram_range=(1, 2), min_df=5),
        representation_model={"KeyBERT": KeyBERTInspired()} if with_keybert else None,
        min_topic_size=MIN_TOPIC_SIZE,
        calculate_probabilities=False,
    )


log.info("Addestramento BERTopic su %d commenti...", len(train_docs))
topic_model = make_topic_model(TOPIC_SEED)
# Addestramento sul campione bilanciato, poi assegnazione a tutti i commenti sopra soglia
train_topics, _ = topic_model.fit_transform(train_docs, train_embeddings)
train_topics = np.array(train_topics)
log.info("Topic: %d, outlier nel campione: %.1f%%", len(set(train_topics)) - 1, (train_topics == -1).mean() * 100)

log.info("Assegnazione dei topic a %d commenti...", len(eligible_docs))
eligible_topics, _ = topic_model.transform(eligible_docs, eligible_embeddings)
eligible["topic"] = np.array(eligible_topics)

### 5.6 Outlier

In questa prima esecuzione gli outlier (topic −1, commenti che HDBSCAN non assegna a nessun cluster abbastanza denso) restano tali. La cella misura quanti sono, in quali periodi, a quali lunghezze e per commenti e risposte. La riga "10-11 parole" serve a verificare la soglia del §5.2.

In [ ]:
eligible["outlier"] = eligible["topic"] == -1
# Fasce di lunghezza: la prima (10-11) verifica la soglia MIN_WORDS del §5.2
length_bins = pd.cut(eligible["word_count_clean"], bins=[9, 11, 14, 19, 29, 49, 99, 10_000],
                     labels=["10-11", "12-14", "15-19", "20-29", "30-49", "50-99", "100+"])
display(pd.Series({
    "commenti con topic assegnato": int((~eligible["outlier"]).sum()),
    "outlier": int(eligible["outlier"].sum()),
    "outlier %": round(eligible["outlier"].mean() * 100, 1),
    "outlier nel campione di addestramento %": round((train_topics == -1).mean() * 100, 1),
}))
display(eligible.groupby("period", observed=True)["outlier"].mean().mul(100).round(1).to_frame("outlier %").T)
display(eligible.groupby(length_bins, observed=True)["outlier"].agg(["size", "mean"])
        .rename(columns={"size": "commenti", "mean": "outlier %"}).assign(**{"outlier %": lambda x: (x["outlier %"] * 100).round(1)}).T)
eligible.groupby("is_reply")["outlier"].mean().mul(100).round(1).rename({False: "commenti", True: "risposte"}).to_frame("outlier %").T

- 148.465 commenti su 268.866 (55,2%) restano outlier; nel campione di addestramento erano il 50,0%.
- Per periodo gli outlier vanno dal 47,7% (2021-2024) al 57,4% (2025-presente). Il 2025-presente, il periodo più sottorappresentato nel campione (27,4% contro il 68,6% dei commenti), è anche quello con più outlier: parte dei suoi commenti non trova un topic addestrato su pochi esempi di quel periodo.
- Per lunghezza: 63,6% tra 10 e 11 parole, 63,0% tra 12 e 14, poi la quota scende fino al 52,6% oltre le 100 parole. La fascia 10-11 si comporta come la 12-14, e la soglia MIN_WORDS = 10 è confermata (§5.2). Anche i commenti lunghi restano outlier in più di metà dei casi, verosimilmente perché trattano più temi insieme o esprimono opinioni generali senza un tema dominante.
- Commenti all'articolo e risposte hanno quote quasi uguali (55,4% e 55,0%).

In [ ]:
# Pipeline del §5; i numeri sono letti dalle variabili del §5.1-§5.6
n_topics = len(set(train_topics) - {-1})
plot_flow([
    ("All comments", f"{len(comments):,}", f"mentions, names\nand URLs removed\n({changed.mean():.1%} changed)"),
    (f">= {MIN_WORDS} words", f"{len(eligible):,}", f"{comments['topic_eligible'].mean():.1%} of comments;\nshorter ones are\nreactions, no topic"),
    ("Embeddings", f"{all_embeddings.shape[1]} dims", f"{EMBEDDING_MODEL},\nall comments, cached\nin chunks of {EMBEDDING_CHUNK:,}"),
    ("Balanced sample", f"{len(train):,}", f"at most {N_PER_PERIOD:,}\ncomments per period\n(seed {SAMPLE_SEED})"),
    ("BERTopic fit", f"{n_topics} topics", f"UMAP 5d -> HDBSCAN\n(min_topic_size {MIN_TOPIC_SIZE})\n-> c-TF-IDF + KeyBERT"),
    ("transform", f"{eligible['outlier'].mean():.1%} outliers", f"topic for all\n{len(eligible):,} comments;\n4-seed check (§5.8)"),
], "Comment topic modeling pipeline (§5)", "comments_topic_pipeline.png")


La pipeline dei topic dal corpus completo all'assegnazione finale: pulizia, soglia di lunghezza, embedding, campione bilanciato, addestramento e `transform`. Ogni riquadro riporta il numero di commenti o il risultato del passaggio, letti dalle variabili del notebook. Lo schema è salvato in `docs/figures/comments_topic_pipeline.png`.

### 5.7 Topic e parole caratterizzanti

Due output: la tabella dei topic (dimensione nel campione e su tutti i commenti, parole c-TF-IDF e KeyBERT) e due commenti rappresentativi per topic, troncati. La quota dei topic per periodo e la relativa heatmap verrà prodotta dopo l'accorpamento e la riassegnazione degli outlier (§5.12).

In [ ]:
# Tabella dei topic (outlier esclusi) e assegnazione per commento, entrambe salvate su disco
info = topic_model.get_topic_info()
info = info[info["Topic"] != -1].copy()
info["parole_ctfidf"] = info["Topic"].map(lambda t: ", ".join(w for w, _ in topic_model.get_topic(t)[:8]))
info["parole_keybert"] = info["Topic"].map(lambda t: ", ".join(w for w, _ in topic_model.get_topic(t, full=True)["KeyBERT"][:8]))
info["commenti_campione"] = info["Count"]
info["commenti_tutti"] = info["Topic"].map(eligible["topic"].value_counts()).fillna(0).astype(int)
topic_info = info[["Topic", "commenti_campione", "commenti_tutti", "parole_ctfidf", "parole_keybert"]].rename(columns={"Topic": "topic"})
topic_info.to_csv(OUT_COMMENT_TOPIC_INFO, index=False)
log.info("Scritto %s (%d topic)", OUT_COMMENT_TOPIC_INFO.relative_to(BASE_DIR), len(topic_info))

eligible[["comment_id", "article_id", "period", "topic"]].to_parquet(OUT_COMMENT_TOPICS, index=False)
log.info("Scritto %s", OUT_COMMENT_TOPICS.relative_to(BASE_DIR))

with pd.option_context("display.max_colwidth", 90, "display.max_rows", 200):
    display(topic_info)

Una riga per topic, ordinati per dimensione nel campione. commenti_campione: commenti del campione di addestramento assegnati al topic; commenti_tutti: commenti assegnati dopo transform su tutti i commenti sopra soglia; parole_ctfidf e parole_keybert: le 8 parole caratterizzanti nelle due rappresentazioni.

- Il rapporto tra le due colonne di conteggio indica il periodo del topic. Un topic tipico del 2025-presente cresce molto con transform, perché quel periodo pesa il 68,6% dei commenti ma solo il 27,4% del campione: il topic 0 passa da 1.939 a 14.687 commenti (×7,6). Un topic tipico del 2003-2016 cresce poco: il 35 (Arizona, legge federale) passa da 219 a 275.

In [ ]:
# Due commenti rappresentativi per topic (dal campione di addestramento), troncati
labels = topic_info.set_index("topic")["parole_ctfidf"].str.split(", ").str[:3].str.join(", ")
for t in topic_info["topic"].head(40):
    docs = topic_model.get_representative_docs(t)[:2]
    print(f"\n[{t}] {labels.get(t, '')}")
    for d in docs:
        print("   -", d[:160])

### 5.8 Stabilità rapida sui seed

I commenti sono testi brevi: prima di interpretarne i topic si controlla che non dipendano dal seed di UMAP. La pipeline viene rieseguita sullo stesso campione con 4 seed (senza KeyBERT, che cambia le parole ma non l'assegnazione) e le partizioni sono confrontate a coppie con l'**Adjusted Mutual Information** (1 = stesse partizioni, 0 = non più simili del caso), su tutti i commenti del campione e sul solo nucleo (commenti non outlier in entrambe le esecuzioni). Si considera in collasso un topic che raccoglie più del 50% del campione.

In [ ]:
from itertools import combinations
from sklearn.metrics import adjusted_mutual_info_score

# Il seed 42 e' il modello di riferimento gia' addestrato; gli altri tre si addestrano qui (circa 1 minuto ciascuno)
STABILITY_SEEDS = [42, 0, 1, 7]
seed_topics = {TOPIC_SEED: train_topics}
for seed in STABILITY_SEEDS:
    if seed not in seed_topics:
        log.info("Seed %d...", seed)
        model = make_topic_model(seed, with_keybert=False)
        seed_topics[seed] = np.array(model.fit_transform(train_docs, train_embeddings)[0])

seed_table = pd.DataFrame([
    {
        "seed": seed,
        "topic": len(set(t) - {-1}),
        "outlier %": round((t == -1).mean() * 100, 1),
        "topic più grande %": round(pd.Series(t[t != -1]).value_counts().max() / len(t) * 100, 1),
    }
    for seed, t in seed_topics.items()
]).set_index("seed")
display(seed_table)

ami_all, ami_core = [], []
for a, b in combinations(STABILITY_SEEDS, 2):
    ta, tb = seed_topics[a], seed_topics[b]
    ami_all.append(adjusted_mutual_info_score(ta, tb))
    core = (ta != -1) & (tb != -1)
    ami_core.append(adjusted_mutual_info_score(ta[core], tb[core]))
print(f"Collassi (un topic > 50%): {(seed_table['topic più grande %'] > 50).sum()}/{len(seed_table)}")
print(f"AMI tutti i commenti: media {np.mean(ami_all):.3f} (min {min(ami_all):.3f})")
print(f"AMI nucleo:           media {np.mean(ami_core):.3f} (min {min(ami_core):.3f})")

La tabella riporta per ogni seed il numero di topic, la quota di outlier e la dimensione del topic più grande (in % del campione). Sotto, i collassi e l'AMI medio e minimo sulle 6 coppie di seed.

- 49 topic con tre seed su quattro, 52 con il seed 7; outlier tra il 47,8% e il 50,0%; il topic più grande non supera il 7,5%: nessun collasso.
- AMI su tutti i commenti del campione 0,796 (minimo 0,776), sul nucleo 0,977 (minimo 0,967). La differenza tra le due misure viene dagli outlier: quali commenti finiscono tra gli outlier cambia un po' da un seed all'altro, mentre i commenti classificati in entrambe le esecuzioni finiscono quasi sempre negli stessi gruppi.

### 5.9 Cosa dicono i risultati (prima esecuzione)

- **49 topic**, tutti leggibili: dall'operato degli agenti (topic 0, il più grande: agenti, polizia, "abolish ICE") a lavoro e salari, deportazioni di Obama e Trump, Biden e il confine, detenzione e prigioni private, bambini e famiglie, Corte Suprema, asilo, proteste di Portland, fede cristiana, sanità, bilancio, critica al NYT, fino a temi di nicchia (patenti e documenti, Statua della Libertà, persone transgender in detenzione).
- **Stabili**: 49-52 topic su 4 seed, nessun collasso, AMI 0,80 su tutti i commenti e 0,98 sul nucleo non outlier, come con MiniLM: sui commenti BERTopic dà partizioni affidabili.
- **Outlier al 55,2%** sui commenti assegnati con transform (50,0% nel campione di addestramento), più alti nel 2025-presente (57,4%) che negli altri periodi (48-52%), uguali tra commenti e risposte (55%). Più alti nei commenti brevi (63-64% sotto le 15 parole) ma non trascurabili nemmeno in quelli lunghi (53%). Sono il punto da affrontare nella seconda esecuzione (riassegnazione per similarità di embedding o c-TF-IDF, come reduce_outliers nel notebook 04).
- **Sovrapposizioni da rivedere**: alcuni topic sono varianti dello stesso tema (17 e 18 critiche al NYT; 8 e 24 democratici e repubblicani; 23 e 45 muro e confine; 5 e 38 immigrazione illegale; 1 e 29 lavoro e datori di lavoro; 31 e 33 previdenza sociale). Si possono accorpare con reduce_topics o merge_topics dopo aver deciso come trattare gli outlier.

### 5.10 Seconda esecuzione: accorpamento dei topic quasi duplicati

La prima esecuzione (§5.5-6.10) lascia due problemi: topic che sono varianti dello stesso tema e più di metà dei commenti tra gli outlier. La seconda esecuzione li affronta in quest'ordine: prima l'accorpamento, poi la riassegnazione degli outlier (§5.11), così gli outlier vengono confrontati con i topic definitivi.

**Coppie accorpate** (scelte sulle parole caratterizzanti e sui commenti rappresentativi del §5.7):

| Topic | Tema comune |
|---|---|
| 17 + 18 | critiche al NYT |
| 8 + 24 | democratici e repubblicani |
| 23 + 45 | muro e confine |
| 5 + 38 | immigrazione illegale |

**Come funziona.** merge_topics unisce i topic sul campione di addestramento: ricalcola le parole c-TF-IDF e KeyBERT sui commenti uniti e prende come embedding del nuovo topic la media degli embedding dei due topic pesata per dimensione. BERTopic poi **rinumera tutti i topic** in ordine di dimensione: la tabella sotto riporta per ogni topic nuovo i numeri della prima esecuzione. Le assegnazioni del transform (tutti i commenti sopra soglia) sono tradotte con la stessa corrispondenza, senza rieseguire il modello.

Le parole caratterizzanti restano calcolate sul campione bilanciato, come nella prima esecuzione: ricalcolarle su tutti i 268.866 commenti le farebbe dipendere quasi solo dal 2025-presente (68,6% dei commenti).

In [ ]:
# Coppie scelte dall'utente (numeri della prima esecuzione, §5.7)
MERGE_PAIRS = [[17, 18], [8, 24], [23, 45], [5, 38]]

first_run_info = topic_info.set_index("topic")
topic_model.merge_topics(train_docs, MERGE_PAIRS)
# Corrispondenza numero della prima esecuzione -> numero dopo l'accorpamento (BERTopic rinumera per dimensione)
merge_map = {int(k): int(v) for k, v in topic_model.topic_mapper_.get_mappings(original_topics=False).items()}
assert all(merge_map[a] == merge_map[b] for a, b in MERGE_PAIRS)

eligible["topic_merged"] = eligible["topic"].map(merge_map).astype(int)
train_topics_merged = np.array([merge_map[t] for t in train_topics])
log.info("Topic dopo l'accorpamento: %d (erano %d)", len(set(merge_map.values()) - {-1}), len(first_run_info))

merged_from = pd.Series(merge_map).drop(-1).groupby(lambda t: merge_map[t]).apply(lambda s: list(s.index))
merged_info = pd.DataFrame({
    "da_prima_esecuzione": merged_from.map(lambda ts: " + ".join(map(str, ts))),
    "commenti_campione": pd.Series(train_topics_merged).value_counts(),
    "commenti_tutti": eligible["topic_merged"].value_counts(),
    "parole_ctfidf": pd.Series({t: ", ".join(w for w, _ in topic_model.get_topic(t)[:8]) for t in merged_from.index}),
}).loc[merged_from.index].rename_axis("topic")
with pd.option_context("display.max_colwidth", 90):
    display(merged_info[merged_info["da_prima_esecuzione"].str.contains(r"\+")])

Tabella con solo i quattro topic nati dall'accorpamento: numero nuovo (indice), topic della prima esecuzione da cui vengono, commenti nel campione e su tutti i commenti sopra soglia, nuove parole caratterizzanti.

### 5.11 Riassegnazione degli outlier

**Metodo e soglia.** Come nel notebook 04 (§5.3): reduce_outliers(strategy="embeddings") confronta l'embedding di ogni outlier con l'embedding di ogni topic (similarità coseno) e lo assegna al topic più simile solo se la similarità è almeno 0,5; sotto soglia il commento resta outlier. La soglia è più alta di quella degli articoli (0,4 nel notebook 04): con 0,4 verrebbe riassegnata quasi tutta la coda degli outlier, compresi commenti generici che somigliano poco al topic in cui finirebbero, e troppi commenti inutili entrerebbero nei topic. Con 0,5 si riassegnano solo gli outlier la cui similarità cade nella fascia dei commenti già assegnati da HDBSCAN (grafico sotto: distribuzione arancione contro blu).

**Controllo.** La stessa soglia non ha per forza lo stesso effetto su commenti brevi e informali e su articoli. Il grafico mette a confronto due distribuzioni:

- per gli **outlier**, la similarità con il topic più vicino (quella che decide la riassegnazione);
- per i **commenti già assegnati** da HDBSCAN, la similarità con il proprio topic: è il riferimento di cosa significa "appartenere a un topic".

Sotto il grafico, la quota di outlier che resta con le soglie 0,3, 0,4 e 0,5, in totale e per periodo, solo come analisi di sensibilità.

I commenti riassegnati sono segnati con topic_reassigned = True: la loro appartenenza al topic è più debole di quella decisa da HDBSCAN e nelle analisi successive si può controllare che i risultati tengano anche sui soli commenti non riassegnati.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

OUTLIER_THRESHOLD = 0.5  # più alta del notebook 04 (0.4): vedi grafico delle due distribuzioni
SENSITIVITY_THRESHOLDS = [0.3, 0.4, 0.5]

# topic_embeddings_ ha la riga degli outlier in testa (_outliers = 1)
topic_vectors = topic_model.topic_embeddings_[topic_model._outliers:]
is_out = (eligible["topic_merged"] == -1).values
sim_out = cosine_similarity(eligible_embeddings[is_out], topic_vectors)
best_sim = sim_out.max(axis=1)
best_topic = sim_out.argmax(axis=1)

assigned_mask = ~is_out
own_sim = np.einsum("ij,ij->i",
                    eligible_embeddings[assigned_mask] / np.linalg.norm(eligible_embeddings[assigned_mask], axis=1, keepdims=True),
                    (v := topic_vectors[eligible.loc[assigned_mask, "topic_merged"].values]) / np.linalg.norm(v, axis=1, keepdims=True))

fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.linspace(-0.1, 1, 56)
ax.hist(own_sim, bins=bins, density=True, alpha=0.55, color="#4c72b0", label=f"assigned by HDBSCAN: similarity to own topic (n={assigned_mask.sum():,})")
ax.hist(best_sim, bins=bins, density=True, alpha=0.55, color="#dd8452", label=f"outliers: similarity to closest topic (n={is_out.sum():,})")
for t in SENSITIVITY_THRESHOLDS:
    ax.axvline(t, color="#333" if t == OUTLIER_THRESHOLD else "#999", ls="-" if t == OUTLIER_THRESHOLD else "--", lw=1)
ax.text(OUTLIER_THRESHOLD + 0.01, ax.get_ylim()[1] * 0.92, f"threshold {OUTLIER_THRESHOLD}", fontsize=8)
ax.set_xlabel("cosine similarity between comment and topic embedding")
ax.set_ylabel("density")
ax.legend(fontsize=7, loc="upper left")
ax.set_title("Outlier reassignment: similarity to topics", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_outlier_similarity.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "comments_outlier_similarity.png", dpi=200, bbox_inches="tight")
plt.show()

def remaining_outliers(threshold: float) -> pd.Series:
    still_out = np.zeros(len(eligible), dtype=bool)
    still_out[is_out] = best_sim < threshold
    return pd.Series(still_out).groupby(eligible["period"].values, observed=True).mean().mul(100)

sens = pd.DataFrame({f"soglia {t}": remaining_outliers(t) for t in SENSITIVITY_THRESHOLDS}).reindex(period_labels)
sens.loc["tutti"] = [(best_sim < t).sum() / len(eligible) * 100 for t in SENSITIVITY_THRESHOLDS]
display(sens.round(1).T.rename_axis("outlier residui %"))
print(f"Similarità con il proprio topic (assegnati): mediana {np.median(own_sim):.2f}, 10° percentile {np.percentile(own_sim, 10):.2f}")
print(f"Assegnati da HDBSCAN con similarità sotto la soglia {OUTLIER_THRESHOLD}: {(own_sim < OUTLIER_THRESHOLD).mean():.1%}")
print(f"Similarità con il topic più vicino (outlier): mediana {np.median(best_sim):.2f}")

In blu la similarità dei commenti assegnati da HDBSCAN con il proprio topic, in arancione quella degli outlier con il topic più vicino; la linea continua è la soglia usata (0,5), le tratteggiate 0,3 e 0,4. Gli outlier a destra della soglia vengono riassegnati. Se la soglia cade nella parte bassa della distribuzione blu, i commenti riassegnati somigliano al loro topic meno dei commenti assegnati da HDBSCAN: è il motivo del flag topic_reassigned. La tabella riporta la quota di commenti sopra soglia che resterebbe outlier con ciascuna soglia, per periodo e in totale.

- Gli outlier non sono commenti lontani da tutto: la loro similarità con il topic più vicino ha mediana 0,53, poco sotto la mediana dei commenti assegnati con il proprio topic (0,63), e le due distribuzioni si sovrappongono largamente. HDBSCAN li scarta perché non stanno in una zona densa, non perché non somiglino a nessun tema.
- La soglia 0,5 cade nella coda bassa dei commenti assegnati: il 15,0% di questi ha una similarità con il proprio topic sotto 0,5. Un commento riassegnato con similarità 0,55 è quindi più debole della grande maggioranza degli assegnati: per questo il flag.
- La tabella mostra quanto conta la soglia: con 0,3 resterebbe outlier il 2,8% dei commenti, con 0,4 l'8,7%, con 0,5 il 21,8%. Il 2025-presente ha sempre circa il doppio di outlier residui degli altri periodi (25,2% a 0,5), coerente con il fatto che è sottorappresentato nel campione di addestramento (27% del campione, 69% dei commenti) e quindi i topic lo coprono peggio.

In [ ]:
new_topics = topic_model.reduce_outliers(eligible_docs, eligible["topic_merged"].tolist(), strategy="embeddings",
                                         embeddings=eligible_embeddings, threshold=OUTLIER_THRESHOLD)
eligible["topic_final"] = np.array(new_topics)
eligible["topic_reassigned"] = (eligible["topic_merged"] == -1) & (eligible["topic_final"] != -1)
# Controllo: reduce_outliers deve coincidere con il calcolo della cella precedente
expected = np.where(best_sim >= OUTLIER_THRESHOLD, best_topic, -1)
assert (eligible.loc[is_out, "topic_final"].values == expected).all()

display(pd.DataFrame({
    "outlier %": [eligible["topic_merged"].eq(-1).mean() * 100, eligible["topic_final"].eq(-1).mean() * 100],
}, index=["prima della riassegnazione", "dopo"]).round(1).T)
by_period = pd.DataFrame({
    "outlier prima %": eligible.groupby("period", observed=True)["topic_merged"].apply(lambda s: s.eq(-1).mean() * 100),
    "outlier dopo %": eligible.groupby("period", observed=True)["topic_final"].apply(lambda s: s.eq(-1).mean() * 100),
    "riassegnati sul totale con topic %": eligible[eligible["topic_final"] != -1].groupby("period", observed=True)["topic_reassigned"].mean() * 100,
}).reindex(period_labels)
display(by_period.round(1).T)

# Quota di riassegnati per topic: topic fatti soprattutto di riassegnati sono più deboli
final_assigned = eligible[eligible["topic_final"] != -1]
reassigned_share = final_assigned.groupby("topic_final")["topic_reassigned"].mean().mul(100)
labels_final = pd.Series({t: ", ".join(w for w, _ in topic_model.get_topic(t)[:3]) for t in reassigned_share.index})
order = reassigned_share.sort_values()
fig, ax = plt.subplots(figsize=(8, 0.2 * len(order) + 1.5))
ax.barh([f"{t}: {labels_final[t]}" for t in order.index], order.values, color="#dd8452")
ax.axvline(final_assigned["topic_reassigned"].mean() * 100, color="#333", lw=1, ls="--", label="all topics")
ax.set_xlabel("% of the topic's comments that were reassigned outliers")
ax.tick_params(axis="y", labelsize=6)
ax.legend(fontsize=7, loc="lower right")
ax.set_title("Share of reassigned outliers per topic", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_reassigned_share.png", dpi=200, bbox_inches="tight")
plt.show()

# Salvataggio: prima esecuzione (topic), accorpamento (topic_merged), finale (topic_final + flag)
eligible[["comment_id", "article_id", "period", "topic", "topic_merged", "topic_final", "topic_reassigned"]].to_parquet(OUT_COMMENT_TOPICS, index=False)
log.info("Scritto %s (colonne topic, topic_merged, topic_final, topic_reassigned)", OUT_COMMENT_TOPICS.relative_to(BASE_DIR))
final_info = merged_info.assign(
    commenti_tutti_final=final_assigned["topic_final"].value_counts(),
    riassegnati_pct=reassigned_share.round(1),
    parole_keybert=pd.Series({t: ", ".join(w for w, _ in topic_model.get_topic(t, full=True)["KeyBERT"][:8]) for t in merged_info.index}),
).reset_index()
OUT_COMMENT_TOPIC_INFO_FINAL = PROCESSED_DIR / "comments_topic_info_final.csv"
final_info.to_csv(OUT_COMMENT_TOPIC_INFO_FINAL, index=False)
log.info("Scritto %s (%d topic)", OUT_COMMENT_TOPIC_INFO_FINAL.relative_to(BASE_DIR), len(final_info))

La prima tabella dà la quota di outlier prima e dopo la riassegnazione con soglia 0,5; la seconda la stessa quota per periodo e, tra i commenti che hanno un topic dopo la riassegnazione, quanti sono riassegnati. Nel grafico, per ogni topic finale, la quota dei suoi commenti che erano outlier; la linea tratteggiata è la media di tutti i topic. Un topic con una quota molto sopra la media è fatto soprattutto di assegnazioni deboli.

- Gli outlier scendono dal 55,2% al 21,8%, tra l'11,6% e il 15,1% nei periodi fino al 2024 e al 25,2% nel 2025-presente.
- Il prezzo è che il 43% dei commenti con un topic è un outlier riassegnato (39-43% per periodo): metà delle assegnazioni finali è debole.
- La quota di riassegnati varia molto tra topic. È bassa nei topic legati a una vicenda o a un lessico preciso (5 Biden e il confine 10%, 22 lingua inglese e spagnola 13%, 0 operato degli agenti 17%, 29 numero di previdenza e identità 18%) e alta nei temi generali di opinione politica, che fanno da calamita: il topic 17 ("trump, president, power") passa da 3.353 a 15.013 commenti, il 78% riassegnati, e diventa il secondo topic più grande; seguono 19 muro e confine (72%), 40 polizia e maschere (71%), 27 datori di lavoro (67%). Il topic 17 finale va letto come un contenitore di commenti sul potere di Trump in generale, non come un tema specifico.

Output: comments_topics.parquet ora contiene, per ogni commento sopra soglia, il topic della prima esecuzione (topic), quello dopo l'accorpamento (topic_merged), quello finale (topic_final) e il flag topic_reassigned; comments_topic_info_final.csv ha i topic finali con la corrispondenza alla prima esecuzione. Le quote per periodo con i topic finali sono nel §5.12.

### 5.12 Topic finali per periodo

Il primo grafico mostra la quota di ogni topic tra i commenti del periodo che hanno un topic (colonne che sommano a 100), con i 45 topic finali. Il confronto con la prima esecuzione si fa sugli stessi topic: il secondo grafico mostra, per ogni topic e periodo, quanto cambia la quota passando dai soli commenti assegnati da HDBSCAN (topic_merged, cioè la prima esecuzione con le coppie già accorpate) ai topic finali con gli outlier riassegnati (topic_final). Così si vede se la riassegnazione sposta il quadro per periodo o lo lascia com'era.

In [ ]:
final_assigned = eligible[eligible["topic_final"] != -1]
hdbscan_assigned = eligible[eligible["topic_merged"] != -1]
share_final = pd.crosstab(final_assigned["topic_final"], final_assigned["period"], normalize="columns").mul(100).reindex(columns=period_labels)
share_hdbscan = (pd.crosstab(hdbscan_assigned["topic_merged"], hdbscan_assigned["period"], normalize="columns").mul(100)
                 .reindex(index=share_final.index, columns=period_labels).fillna(0))
row_labels = [f"{t}: {labels_final[t]}" for t in share_final.index]

fig, ax = plt.subplots(figsize=(8, 0.28 * len(share_final) + 1.5))
im = ax.imshow(share_final.values, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(period_labels)), period_labels)
ax.set_yticks(range(len(share_final)), row_labels, fontsize=7)
ax.grid(False)
for (r, c), v in np.ndenumerate(share_final.values):
    if v >= 1:
        ax.text(c, r, f"{v:.0f}", ha="center", va="center", fontsize=6, color="white" if v > share_final.values.max() / 2 else "#333")
fig.colorbar(im, ax=ax, label="% of the period's comments with a topic")
ax.set_title("Final comment topics by period (merged + outliers reassigned)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_topics_final_by_period.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "comments_topics_final_by_period.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

# Variazione in punti percentuali: finale meno solo-HDBSCAN, stessi topic
delta = share_final - share_hdbscan
lim = np.abs(delta.values).max()
fig, ax = plt.subplots(figsize=(8, 0.28 * len(delta) + 1.5))
im = ax.imshow(delta.values, aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim)
ax.set_xticks(range(len(period_labels)), period_labels)
ax.set_yticks(range(len(delta)), row_labels, fontsize=7)
ax.grid(False)
for (r, c), v in np.ndenumerate(delta.values):
    if abs(v) >= 1:
        ax.text(c, r, f"{v:+.0f}", ha="center", va="center", fontsize=6, color="white" if abs(v) > lim / 2 else "#333")
fig.colorbar(im, ax=ax, label="change in share (percentage points)")
ax.set_title("Effect of outlier reassignment on topic shares by period", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_topics_reassignment_delta.png", dpi=200, bbox_inches="tight")
plt.show()

# Riepilogo numerico per la lettura: tema dominante per periodo prima e dopo, spostamento complessivo
summary = pd.DataFrame({
    "topic dominante (solo HDBSCAN)": [f"{share_hdbscan[p].idxmax()} ({share_hdbscan[p].max():.0f}%)" for p in period_labels],
    "topic dominante (finale)": [f"{share_final[p].idxmax()} ({share_final[p].max():.0f}%)" for p in period_labels],
    "variazione totale (somma |Δ| / 2, pp)": [delta[p].abs().sum() / 2 for p in period_labels],
    "Spearman tra le quote": [share_hdbscan[p].corr(share_final[p], method="spearman") for p in period_labels],
}, index=period_labels)
display(summary.round(2))
print("Topic con le variazioni maggiori (pp):")
print(delta.stack().sort_values(key=abs, ascending=False).head(8).round(1).to_string())

- L'ordine dei topic dentro ogni periodo cambia poco (Spearman 0,94-0,95), ma si spostano 13-16 punti di quota per periodo e **il topic dominante cambia in 2 periodi su 4**: detenzione e prigioni → immigrazione illegale nel 2003-2016, bambini e famiglie → Obama contro Trump nel 2017-2020. Nel 2021-2024 Biden e il confine resta primo ma scende dal 16% all'11%, nel 2025-presente l'operato degli agenti resta primo ma scende dal 15% all'11%.
- La direzione è sempre la stessa: **i temi distintivi di ciascun periodo perdono peso** (agenti −5 punti nel 2025-presente, Biden −6 nel 2021-2024, famiglie −3 nel 2017-2020, detenzione −3 nel 2003-2016) e **i temi generali ne guadagnano** (17 Trump e il potere +3/+5 dal 2017, 3 immigrazione illegale +4 nel 2003-2016, 19 muro +1/+3, 4 Obama contro Trump +2). Gli outlier sono soprattutto commenti di opinione politica generale, e la riassegnazione li manda verso i topic generali.
- Conseguenza pratica: per dire di cosa si parla *di specifico* in un periodo valgono le quote dei soli commenti assegnati da HDBSCAN (colonna topic_merged, secondo grafico); i topic finali servono quando ogni commento deve avere un topic (per esempio il sentiment per topic), con il controllo sui soli commenti non riassegnati.

## 6. Quali topic degli articoli generano più dibattito

**Domanda.** Su quali temi giornalistici i lettori discutono di più *tra loro*? Non conta quanti commenti riceve un articolo (il volume, §2-§3), ma quanto i commenti diventano conversazione: risposte a risposte, catene lunghe, lettori che entrano nella discussione.

**Topic dell'articolo.** Ogni articolo ha il topic assegnato nel notebook 04 (BERTopic su titolo, abstract e attacco, embedding mpnet, 25 topic; `articles_ice_topics.parquet`). Gli articoli outlier del notebook 04 (topic −1) sono esclusi.

**Scelte.**
- **Unità = articolo**, non commento: ogni articolo pesa uno, altrimenti i pochi articoli con migliaia di commenti deciderebbero il risultato.
- **Solo articoli con catene registrate** (tutti i commenti dopo `THREADED_REPLIES_START`, aprile 2018, §4): prima il NYT non registrava le risposte a risposte. Il 2003-2016 resta fuori.
- **Almeno 50 commenti per articolo** (`MIN_COMMENTS_DEBATE`): sotto, le quote di un articolo dipendono da pochi commenti.
- **Topic analizzati**: un topic entra se ha almeno 10 articoli (`MIN_TOPIC_ARTICLES`) **oppure** se in almeno un periodo raccoglie almeno il 5% degli articoli del periodo (`MIN_PERIOD_SHARE`). La seconda regola tiene i temi caratteristici di un periodo anche quando, sul totale, hanno pochi articoli.

**Misure di dibattito** (per articolo, strutturali: dicono che i lettori si rispondono, non che sono in disaccordo; il tono è nel notebook 11):
- **quota di risposte in catena**: commenti con `chain_depth ≥ 3` sul totale (§4);
- **profondità media delle risposte**: `chain_depth` medio delle sole risposte (2 = tutte risposte dirette ai commenti);
- **quota di commenti che ricevono risposta**: commenti da cui parte almeno una risposta;
- **quota di lettori nelle catene**: lettori distinti che scrivono almeno una risposta in catena, sul totale dei lettori dell'articolo.

### 6.1 Misure per articolo e topic analizzati

In [ ]:
# Topic degli articoli dal notebook 04 (mpnet, 25 topic; -1 = outlier)
IN_ARTICLE_TOPICS = PROCESSED_DIR / "articles_ice_topics.parquet"
OUT_DEBATE = PROCESSED_DIR / "comments_debate_by_article.csv"
MIN_COMMENTS_DEBATE = 50
MIN_TOPIC_ARTICLES = 10
MIN_PERIOD_SHARE = 0.05

article_topics = pd.read_parquet(IN_ARTICLE_TOPICS, columns=["id", "topic", "topic_name"]).rename(
    columns={"id": "article_id", "topic": "article_topic", "topic_name": "article_topic_name"})

# Solo articoli i cui commenti sono tutti nell'era con catene registrate (thread_stats["threaded"], §4)
threaded_articles = thread_stats.loc[thread_stats["threaded"], "article_id"]
debate_comments = comments[comments["article_id"].isin(threaded_articles)]
has_reply = debate_comments["comment_id"].isin(set(debate_comments["parent_id"].dropna().astype("int64")))
users_in_chains = debate_comments[debate_comments["deep"]].groupby("article_id")["user_key"].nunique()

debate = debate_comments.groupby("article_id").agg(
    commenti=("comment_id", "size"),
    utenti=("user_key", "nunique"),
    quota_in_catena=("deep", "mean"),
)
debate["profondita_media_risposte"] = debate_comments[debate_comments["is_reply"]].groupby("article_id")["chain_depth"].mean()
debate["quota_con_risposta"] = has_reply.groupby(debate_comments["article_id"]).mean()
debate["quota_utenti_in_catena"] = (users_in_chains / debate["utenti"]).fillna(0)
debate = (debate.reset_index()
          .merge(thread_stats[["article_id", "period", "article_pub_date", "section_name", "headline"]], on="article_id")
          .merge(article_topics, on="article_id"))
debate = debate[(debate["commenti"] >= MIN_COMMENTS_DEBATE) & (debate["article_topic"] != -1)].copy()
debate["period"] = debate["period"].astype(str)
debate.to_csv(OUT_DEBATE, index=False)
log.info("Scritto %s (%d articoli)", OUT_DEBATE.relative_to(BASE_DIR), len(debate))

# Selezione dei topic: abbastanza articoli in totale, oppure quota alta in almeno un periodo
debate_periods = [p for p in period_labels if p in set(debate["period"])]
counts = pd.crosstab(debate["article_topic"], debate["period"]).reindex(columns=debate_periods, fill_value=0)
period_share = counts / counts.sum()
topic_names = article_topics.drop_duplicates("article_topic").set_index("article_topic")["article_topic_name"]
selection = pd.DataFrame({
    "articoli": counts.sum(axis=1),
    "quota_max_periodo": period_share.max(axis=1),
})
selection["per_numero"] = selection["articoli"] >= MIN_TOPIC_ARTICLES
selection["per_periodo"] = selection["quota_max_periodo"] >= MIN_PERIOD_SHARE
selection["tenuto"] = selection["per_numero"] | selection["per_periodo"]
selected_topics = selection.index[selection["tenuto"]].tolist()
log.info("Articoli analizzabili: %d; topic tenuti: %d su %d (solo per la regola del periodo: %d)",
         len(debate), len(selected_topics), len(selection),
         int((selection["per_periodo"] & ~selection["per_numero"]).sum()))

# Grafico della selezione: quota degli articoli di ogni periodo per topic; etichette grigie = topic esclusi
order = selection.sort_values("articoli", ascending=False).index
values = period_share.loc[order].values * 100
fig, ax = plt.subplots(figsize=(8, 0.3 * len(order) + 1.6))
im = ax.imshow(values, aspect="auto", cmap="Blues", vmin=0)
ax.set_xticks(range(len(debate_periods)), [f"{p}\n({counts[p].sum()} articles)" for p in debate_periods], fontsize=8)
ax.set_yticks(range(len(order)), [f"{topic_names[t]} (n={selection.loc[t, 'articoli']})" for t in order], fontsize=7.5)
for tick, t in zip(ax.get_yticklabels(), order):
    tick.set_color("#1a1a1a" if selection.loc[t, "tenuto"] else "#aaaaaa")
ax.grid(False)
for (r, c), v in np.ndenumerate(values):
    if v >= 1:
        bold = v >= MIN_PERIOD_SHARE * 100
        ax.text(c, r, f"{v:.0f}", ha="center", va="center", fontsize=6.5,
                color="white" if v > values.max() / 2 else "#333", fontweight="bold" if bold else "normal")
fig.colorbar(im, ax=ax, label="% of the period's articles", shrink=0.6)
ax.set_title(f"Article topics in the debate analysis (≥ {MIN_COMMENTS_DEBATE} comments, from Apr 2018)\n"
             f"kept: ≥ {MIN_TOPIC_ARTICLES} articles or ≥ {MIN_PERIOD_SHARE:.0%} of a period (bold); grey = excluded",
             loc="left", fontsize=9.5)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "debate_topic_selection.png", dpi=200, bbox_inches="tight")
plt.show()

**Come leggere il grafico.** Una riga per topic dell'articolo (tra parentesi gli articoli analizzabili), una colonna per periodo: ogni cella è la quota degli articoli del periodo che appartengono al topic; i numeri in grassetto superano il 5%. Le etichette nere sono i topic analizzati, quelle grigie i topic esclusi. Il 2003-2016 manca perché non ha catene registrate.

- **466 articoli analizzabili**: 365 del 2025-presente, 76 del 2017-2020, 25 del 2021-2024. L'analisi descrive quindi soprattutto il periodo più recente.
- **16 topic su 25** hanno almeno 10 articoli. La regola del periodo non aggiunge nessun topic: gli esclusi non superano mai il 4% di un periodo, e nel 2021-2024 il 4% corrisponde a un solo articolo. La soglia del 5% è quindi sicura (al 4,5% il risultato non cambia).
- Alcuni temi sono legati a un periodo: bambini e famiglie (22% degli articoli del 2017-2020), politica nazionale (36% del 2021-2024), deportazioni verso paesi terzi (16% del 2021-2024), raid e paura nelle comunità (12% del 2025-presente). Per questo il confronto tra topic va controllato per periodo (§6.3).

### 6.2 Dibattito per topic dell'articolo

In [ ]:
# Mediana per topic con intervallo bootstrap al 95% (ricampionamento degli articoli)
DEBATE_MEASURES = {
    "quota_in_catena": "share of comments in reply chains",
    "profondita_media_risposte": "mean depth of replies",
    "quota_con_risposta": "share of comments that get a reply",
    "quota_utenti_in_catena": "share of readers in reply chains",
}
N_BOOT = 2000
rng = np.random.default_rng(42)


def median_ci(values: np.ndarray) -> tuple[float, float, float]:
    boots = np.median(rng.choice(values, size=(N_BOOT, len(values)), replace=True), axis=1)
    return np.median(values), *np.percentile(boots, [2.5, 97.5])


sel = debate[debate["article_topic"].isin(selected_topics)]
summary = pd.DataFrame([
    {"article_topic": t, "measure": m, **dict(zip(["median", "lo", "hi"], median_ci(g[m].dropna().values)))}
    for t, g in sel.groupby("article_topic") for m in DEBATE_MEASURES
])
topic_order = (summary[summary["measure"] == "quota_in_catena"].sort_values("median")["article_topic"].tolist())
y = {t: i for i, t in enumerate(topic_order)}

fig, axes = plt.subplots(1, len(DEBATE_MEASURES), figsize=(14, 0.32 * len(topic_order) + 1.8), sharey=True)
for ax, (m, label) in zip(axes, DEBATE_MEASURES.items()):
    s = summary[summary["measure"] == m]
    for row in s.itertuples():
        reliable = selection.loc[row.article_topic, "per_numero"]
        ax.plot([row.lo, row.hi], [y[row.article_topic]] * 2, color=MAIN_COLOR, lw=1.2, alpha=0.6)
        ax.scatter(row.median, y[row.article_topic], s=30, zorder=3, color=MAIN_COLOR,
                   facecolor=MAIN_COLOR if reliable else "white", linewidth=1.3)
    ax.axvline(debate[m].median(), color="#555", ls="--", lw=1)
    ax.set_title(label, loc="left", fontsize=9)
    ax.tick_params(axis="x", labelsize=8)
axes[0].set_yticks(range(len(topic_order)),
                   [f"{topic_names[t]} (n={selection.loc[t, 'articoli']})" for t in topic_order], fontsize=8)
hollow_note = "; hollow = fewer than 10 articles, kept by period share" if (~selection.loc[topic_order, "per_numero"]).any() else ""
fig.suptitle(f"Debate by article topic: median per article with 95% bootstrap CI (dashed = all articles{hollow_note})",
             x=0.01, ha="left", fontsize=10.5)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "debate_by_article_topic.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "debate_by_article_topic.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

**Come leggere il grafico.** Una riga per topic, ordinati per quota di commenti in catena; un pannello per misura. Il punto è la mediana degli articoli del topic, la barra l'intervallo bootstrap al 95% (2.000 ricampionamenti degli articoli), la linea tratteggiata la mediana di tutti i 466 articoli. Due topic si distinguono davvero solo se le barre non si sovrappongono.

- **Le quattro misure dicono la stessa cosa.** Tra articoli, la quota in catena è correlata con la quota di lettori nelle catene (Spearman 0,97) e con la profondità media delle risposte (0,91), meno con la quota di commenti che ricevono risposta (0,72). Con il numero di commenti la correlazione è quasi nulla (0,17 e −0,08): **il dibattito non è il volume**.
- **Più conversazione** sotto gli articoli sull'azione concreta di ICE sulle persone: la stretta su Chicago e altre città (15,3% di commenti in catena, ma con soli 11 articoli e intervallo largo), bambini e famiglie (13,0%), operazioni e arresti (12,6%), Corte Suprema (12,4%), centri di detenzione (12,3%). La mediana di tutti gli articoli è 10,6%.
- **Meno conversazione** sotto le sparatorie degli agenti (6,4%, con l'intervallo tutto sotto la mediana in tutte e quattro le misure), il finanziamento del DHS e lo shutdown (7,4%) e la guida del DHS (7,6%, intervallo sotto la mediana in due misure; è anche il topic con meno commenti che ricevono risposta, 18,7%). Le sparatorie non sono un tema poco commentato (mediana 329 commenti per articolo, contro 399 di tutti gli articoli; un quarto supera i 700), ma i lettori reagiscono all'articolo più che rispondersi tra loro.
- **Cautela.** Gli intervalli si sovrappongono in gran parte: solo gli estremi sono distinguibili. Il periodo e il numero di commenti non sono ancora controllati (§6.3).

### 6.3 Dibattito o volume? Controllo per periodo e numero di commenti

Due verifiche sulla misura principale, la quota di commenti in catena (le altre tre misure le sono strettamente correlate, §6.2).

1. **Dibattito contro volume.** Per ogni topic, commenti per articolo (mediana) contro quota in catena (mediana). Le linee tratteggiate sono le mediane di tutti gli articoli e dividono il piano in quattro quadranti: in basso a destra i topic con molti commenti ma poca conversazione, in alto a sinistra quelli con pochi commenti ma molta conversazione.
2. **Controllo.** Un topic può sembrare conflittuale perché i suoi articoli sono quasi tutti di un periodo (bambini e famiglie: 2017-2020) o perché sono molto commentati. Regressione lineare per articolo:

   quota in catena ~ topic + periodo + log(commenti)

   Per ogni topic si calcola la **quota attesa a parità di periodo e numero di commenti** (standardizzazione: si assegna quel topic a tutti i 466 articoli, lasciando a ciascuno il suo periodo e il suo numero di commenti, e si fa la media delle previsioni), con intervallo al 95% da 1.000 ricampionamenti bootstrap degli articoli. Si confronta con la media osservata del topic: se le due coincidono, il periodo e il volume non spiegano la differenza. La regressione usa le medie, non le mediane del §6.2, quindi i valori sono un po' più alti.

In [ ]:
# Una riga per topic: mediane per articolo
topic_stats = sel.groupby("article_topic").agg(
    articoli=("article_id", "size"),
    commenti=("commenti", "median"),
    quota_in_catena=("quota_in_catena", "median"),
)
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(topic_stats["commenti"], topic_stats["quota_in_catena"] * 100, s=topic_stats["articoli"] * 6,
           color=MAIN_COLOR, alpha=0.6, edgecolor="white", linewidth=0.8)
# Spostamenti manuali delle etichette che si sovrappongono
LABEL_OFFSETS = {"Raids and fear in communities": (6, -9), "National politics: Trump and the parties": (8, -9)}
for t, row in topic_stats.iterrows():
    ax.annotate(topic_names[t], (row["commenti"], row["quota_in_catena"] * 100), xytext=LABEL_OFFSETS.get(topic_names[t], (6, 0)),
                textcoords="offset points", fontsize=7.5, va="center")
ax.axvline(debate["commenti"].median(), color="#777", ls="--", lw=1)
ax.axhline(debate["quota_in_catena"].median() * 100, color="#777", ls="--", lw=1)
ax.set_xscale("log")
ax.set_xlabel("comments per article (median, log scale)")
ax.set_ylabel("% of comments in reply chains (median)")
for x, y, text, ha in [(0.99, 0.02, "many comments, little conversation", "right"),
                       (0.01, 0.97, "few comments, much conversation", "left")]:
    ax.text(x, y, text, transform=ax.transAxes, ha=ha, va="bottom" if y < 0.5 else "top", fontsize=8, color="#777", style="italic")
ax.set_title("Debate vs volume by article topic (bubble size = articles; dashed = all articles)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "debate_vs_volume.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "debate_vs_volume.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

Ogni bolla è un topic (degli articoli): in orizzontale i commenti per articolo (mediana, scala logaritmica), in verticale la quota di commenti in catena (mediana); la dimensione è il numero di articoli. Le linee tratteggiate sono le mediane di tutti i 466 articoli (399 commenti, 10,6% in catena).

- **Molti commenti, poca conversazione** (in basso a destra): proteste nelle città (480 commenti per articolo, 8,3% in catena) e guida del DHS (430, 7,6%). La politica nazionale è sul confine (626 commenti, 10,3%): i suoi articoli sono i più commentati, ma i lettori si rispondono come la media.
- **Pochi commenti, molta conversazione** (in alto a sinistra): centri di detenzione (182 commenti, 12,3%), casi individuali di espulsione, proteste ai centri di detenzione, raid e paura nelle comunità, operazioni e arresti. Sotto articoli meno letti chi commenta discute di più con gli altri.
- **Molti commenti e molta conversazione** (in alto a destra): la stretta su Chicago, bambini e famiglie, Corte Suprema.
- **Pochi commenti e poca conversazione** (in basso a sinistra): finanziamento del DHS, polizia locale e città santuario, sparatorie degli agenti (329 commenti, 6,4%).
- La posizione orizzontale e quella verticale non sono legate: un topic molto commentato non è per questo più discusso.

In [ ]:
# quota in catena ~ topic + periodo + log(commenti), minimi quadrati con numpy (nessuna dipendenza in più)
N_BOOT_REG = 1000
reg = sel[["article_topic", "period", "commenti", "quota_in_catena"]].reset_index(drop=True)
reg_topics = sorted(reg["article_topic"].unique())
reg_periods = sorted(reg["period"].unique())


def design(df: pd.DataFrame, topic: int | None = None) -> np.ndarray:
    """Intercetta + dummy di topic e periodo (prima categoria come riferimento) + log(commenti)."""
    topics = df["article_topic"] if topic is None else pd.Series(topic, index=df.index)
    cols = [np.ones(len(df))]
    cols += [(topics == t).astype(float).values for t in reg_topics[1:]]
    cols += [(df["period"] == q).astype(float).values for q in reg_periods[1:]]
    cols.append(np.log(df["commenti"].values))
    return np.column_stack(cols)


def standardized_means(df: pd.DataFrame) -> np.ndarray:
    # Un campione bootstrap senza qualche topic o periodo non stima quel coefficiente: scartato
    if df["article_topic"].nunique() < len(reg_topics) or df["period"].nunique() < len(reg_periods):
        return np.full(len(reg_topics), np.nan)
    beta, *_ = np.linalg.lstsq(design(df), df["quota_in_catena"].values, rcond=None)
    # Ogni topic assegnato a tutti gli articoli, con il loro periodo e il loro numero di commenti
    return np.array([(design(reg, t) @ beta).mean() for t in reg_topics])


point = standardized_means(reg)
boot = np.array([standardized_means(reg.sample(len(reg), replace=True, random_state=i)) for i in range(N_BOOT_REG)])
adjusted = pd.DataFrame({
    "osservata": reg.groupby("article_topic")["quota_in_catena"].mean().reindex(reg_topics).values,
    "attesa": point,
    "lo": np.nanpercentile(boot, 2.5, axis=0),
    "hi": np.nanpercentile(boot, 97.5, axis=0),
}, index=reg_topics) * 100
adjusted = adjusted.sort_values("attesa")
overall = reg["quota_in_catena"].mean() * 100

fig, ax = plt.subplots(figsize=(9, 0.38 * len(adjusted) + 1.6))
ypos = np.arange(len(adjusted))
ax.hlines(ypos, adjusted["lo"], adjusted["hi"], color=MAIN_COLOR, lw=1.3, alpha=0.6)
ax.scatter(adjusted["attesa"], ypos, color=MAIN_COLOR, s=34, zorder=3, label="adjusted for period and number of comments")
ax.scatter(adjusted["osservata"], ypos, facecolor="white", edgecolor="#777", s=34, zorder=3, label="observed mean")
ax.axvline(overall, color="#555", ls="--", lw=1)
ax.set_yticks(ypos, [f"{topic_names[t]} (n={selection.loc[t, 'articoli']})" for t in adjusted.index], fontsize=8)
ax.set_xlabel("% of comments in reply chains (mean per article)")
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.set_title("Reply-chain share by article topic, observed vs adjusted (95% bootstrap CI; dashed = all articles)",
             loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "debate_adjusted.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "debate_adjusted.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

beta_all, *_ = np.linalg.lstsq(design(reg), reg["quota_in_catena"].values, rcond=None)
log.info("Coefficiente di log(commenti): %.4f (quota in catena per raddoppio dei commenti: %+.2f punti)",
         beta_all[-1], beta_all[-1] * np.log(2) * 100)

Questo grafico confronta la quota di commenti in catena osservata e quella attesa per ogni topic.
Presenta una riga per topic (i 16 analizzati), dal meno discusso in basso al più discusso. Nell'asse delle X troviamo la percentuale di commenti in catena (risposte a risposte) espresse come media per articolo, in particolare il cerchio vuoto indica la media osservata (media semplice degli articoli del topic) e il cerchio pieno blu la media attesa a parità di periodo e di numero di commenti (calcolata con la regressione). La barra blu indica l'intervallo di confidenza al 95% della media attesa (da 1000 ricampionamenti degli articoli) mentre la linea tratteggiata indica la media di tutti gli articoli dei 16 topic.

- Se i due cerchi sono vicini il periodo e il numero di commenti non spiegano le differenze tra topic.
- se la barra sta a destra della linea tratteggiata il topic è più discusso della media e non a caso.
- se sta a sinistra il topic è meno discusso della media.
- la lunghezza della barra dice quanto sia sicura la stima (barre lunghe indicano pochi articoli o articoli molto diversi tra loro).


**Risultato del §6.** I lettori conversano di più sotto gli articoli sull'azione di ICE nelle comunità e sulle famiglie (raid, bambini, operazioni, detenzione), meno sotto quelli istituzionali (guida e finanziamento del DHS, politica nazionale) e sotto le sparatorie degli agenti. Le differenze non dipendono dal periodo né dal volume dei commenti, ma sono di pochi punti (dall'8% al 14% circa) e la misura è strutturale: dice che i lettori si rispondono, non che sono in disaccordo.

## 7. Il topic dell'articolo e i topic dei commenti

**Domanda.** Sotto un articolo di un certo tema, i lettori parlano di quel tema o portano la discussione altrove? Se sotto gli articoli sulle sparatorie degli agenti si discute di immigrazione illegale, di Trump o del NYT, il topic dell'articolo genera dibattito su altri temi.

**Selezione dei dati.** Quattro filtri, applicati in quest'ordine.

1. **Articoli.** Articoli ICE con almeno 50 commenti raccolti e con un topic nel notebook 04 (outlier −1 esclusi), di **tutti i periodi**: qui le catene di risposte non servono, quindi il 2003-2016 rientra (solo il §7.3 usa le catene e si limita all'era in cui sono registrate, dal 2018-04). Risultato: 537 articoli. Il §6 invece partiva dal 2018-04 (466 articoli).
2. **Topic degli articoli** (notebook 04, mpnet, 25 topic). Stessa regola del §6, ricalcolata su questi 537 articoli: un topic resta se ha **almeno 10 articoli** in totale **oppure almeno il 5% degli articoli di un periodo** (la seconda condizione salva i temi piccoli ma tipici di un periodo con pochi articoli). Restano 18 topic su 25. Il grafico di sintesi alla fine del §7.3 usa i 16 topic tenuti sia qui sia nel §6.
3. **Commenti.** Solo commenti con almeno 10 parole dopo la pulizia di menzioni e URL, come nel §5: sotto questa soglia embedding e topic non sono affidabili. La misura del fuori tema (§7.2) usa tutti questi commenti, anche quelli senza topic, perché si basa sulla similarità con l'articolo e non sui topic.
4. **Topic dei commenti** (§5, 45 topic dopo gli accorpamenti). Si usa `topic_merged`, cioè le sole assegnazioni di HDBSCAN: i commenti riassegnati nel §5.11 sono esclusi, perché la riassegnazione spinge i commenti generici verso i topic generali (§5.12) e qui gonfierebbe proprio lo spostamento verso altri temi che si vuole misurare (decisione dell'utente). Gli outlier (−1) sono esclusi dalle matrici del §7.1 e dalle destinazioni del §7.3. Nessuno dei 45 topic viene scartato a monte: la selezione riguarda solo cosa si mostra (nel §7.1 il lift è grigio nelle celle con meno di 30 commenti; nelle destinazioni del §7.3 compaiono solo i topic dei commenti che sono tra le prime 3 destinazioni di almeno un topic dell'articolo, ma ogni riga è calcolata su tutti i 45).

**Unità.** Per ogni articolo si calcola la distribuzione dei suoi commenti tra i topic dei commenti; la distribuzione di un topic dell'articolo è la media delle distribuzioni dei suoi articoli. Altrimenti gli articoli con migliaia di commenti deciderebbero le quote.

**Percorso.** §7.1 quote e lift dei topic dei commenti per topic dell'articolo; §7.2 distanza dei commenti dal loro articolo (similarità degli embedding); §7.3 deriva della discussione lungo le catene di risposte; §7.4 come cambiano nel tempo i topic dei commenti sotto uno stesso topic dell'articolo.

### 7.1 Di cosa parlano i lettori sotto ogni topic dell'articolo

In [ ]:
# Topic dei commenti: solo assegnazioni di HDBSCAN dopo l'accorpamento (topic_merged), come deciso per la §7
MIN_COMMENTS_SPILLOVER = 50
comment_topic_labels = (pd.read_csv(OUT_COMMENT_TOPIC_INFO_FINAL).set_index("topic")["parole_ctfidf"]
                        .str.split(", ").str[:3].str.join(", "))

open_articles = comment_articles.loc[comment_articles["n_comments_collected"] >= MIN_COMMENTS_SPILLOVER, ["article_id", "period"]]
spill = (eligible.loc[eligible["topic_merged"] != -1, ["comment_id", "article_id", "topic_merged"]]
         .merge(open_articles, on="article_id")
         .merge(article_topics, on="article_id"))
spill = spill[spill["article_topic"] != -1]
spill["period"] = spill["period"].astype(str)

# Stessa regola di selezione del §6, ricalcolata sugli articoli di questa sezione
spill_articles = spill.drop_duplicates("article_id")
spill_counts = pd.crosstab(spill_articles["article_topic"], spill_articles["period"])
spill_selection = pd.DataFrame({"articoli": spill_counts.sum(axis=1),
                                "quota_max_periodo": (spill_counts / spill_counts.sum()).max(axis=1)})
spill_selection["tenuto"] = (spill_selection["articoli"] >= MIN_TOPIC_ARTICLES) | (spill_selection["quota_max_periodo"] >= MIN_PERIOD_SHARE)
spill_topics = spill_selection.index[spill_selection["tenuto"]]
log.info("§7: %d articoli, %d commenti con topic; topic dell'articolo tenuti: %d su %d",
         spill["article_id"].nunique(), len(spill), len(spill_topics), len(spill_selection))

# Distribuzione dei commenti di ogni articolo tra i topic dei commenti, poi media per topic dell'articolo
per_article_dist = pd.crosstab(spill["article_id"], spill["topic_merged"], normalize="index")
article_topic_of = spill_articles.set_index("article_id")["article_topic"]
baseline = per_article_dist.mean()                     # tutti gli articoli, ogni articolo pesa uno
by_topic = per_article_dist.groupby(article_topic_of).mean().loc[spill_topics]
lift = by_topic / baseline
# Supporto: commenti di quel topic sotto articoli di quel topic; sotto la soglia il lift non si legge
support = pd.crosstab(spill["article_topic"], spill["topic_merged"]).reindex(index=spill_topics, columns=lift.columns, fill_value=0)
MIN_CELL_COMMENTS = 30

# Colonne ordinate per il topic dell'articolo in cui il topic del commento è più sovrarappresentato: diagonale leggibile
row_order = spill_selection.loc[spill_topics].sort_values("articoli", ascending=False).index
peak_row = lift.loc[row_order].idxmax()
col_order = sorted(lift.columns, key=lambda c: (list(row_order).index(peak_row[c]), -lift.loc[peak_row[c], c]))
row_labels = [f"{topic_names[t]} (n={spill_selection.loc[t, 'articoli']})" for t in row_order]
col_labels = [f"{c}: {comment_topic_labels[c]}" for c in col_order]

fig, axes = plt.subplots(2, 1, figsize=(17, 2 * (0.34 * len(row_order) + 1.2) + 3.2),
                         gridspec_kw={"hspace": 0.08})
share_values = by_topic.loc[row_order, col_order].values * 100
with np.errstate(divide="ignore"):  # lift 0 solo in celle sotto il supporto minimo, poi grigie
    log_lift = np.log2(lift.loc[row_order, col_order].values)
weak = support.loc[row_order, col_order].values < MIN_CELL_COMMENTS
log_lift_shown = np.where(weak, np.nan, log_lift)

im0 = axes[0].imshow(share_values, aspect="auto", cmap="Blues", vmin=0)
for (r, c), v in np.ndenumerate(share_values):
    if v >= 4:
        axes[0].text(c, r, f"{v:.0f}", ha="center", va="center", fontsize=6.5, color="white" if v > share_values.max() / 2 else "#333")
fig.colorbar(im0, ax=axes[0], label="% of the article's comments\n(mean over articles)", pad=0.01, fraction=0.025)
axes[0].set_title("Share of each comment topic under each article topic (numbers where ≥ 4%)", loc="left", fontsize=10)

cmap = plt.get_cmap("RdBu_r").copy()
cmap.set_bad("#eeeeee")
lim = np.nanmax(np.abs(log_lift_shown))
im1 = axes[1].imshow(log_lift_shown, aspect="auto", cmap=cmap, vmin=-lim, vmax=lim)
for (r, c), v in np.ndenumerate(log_lift_shown):
    if not np.isnan(v) and v >= 1:
        axes[1].text(c, r, f"×{2 ** v:.0f}", ha="center", va="center", fontsize=6.5, color="white" if v > lim / 2 else "#333")
fig.colorbar(im1, ax=axes[1], label="lift (log2): 1 = twice the average,\n−1 = half", pad=0.01, fraction=0.025)
axes[1].set_title(f"Lift: share under the article topic / share under all articles (×n where at least double; "
                  f"grey = fewer than {MIN_CELL_COMMENTS} comments)", loc="left", fontsize=10)

for ax in axes:
    ax.set_yticks(range(len(row_order)), row_labels, fontsize=7.5)
    ax.grid(False)
axes[0].set_xticks([])
axes[1].set_xticks(range(len(col_order)), col_labels, rotation=60, ha="right", fontsize=7)
fig.subplots_adjust(top=0.955)
fig.suptitle("Article topic (rows) → what readers talk about in the comments (columns, comment topics)",
             x=0.01, ha="left", fontsize=11, y=0.985)
fig.savefig(PLOTS_DIR / "spillover_topic_matrix.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "spillover_topic_matrix.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

le righe sono i topic dell'articolo (tra parentesi gli articoli), le colonne i topic dei commenti (numero e prime tre parole del §5). Le colonne sono ordinate in modo che ogni topic dei commenti stia vicino al topic dell'articolo sotto cui è più sovrarappresentato, per questo compare una diagonale.
- primo grafico: per ogni topic dell'articolo, quanta parte dei commenti di un suo articolo tipico appartiene a ciascun topic dei commenti (media sugli articoli, righe che sommano a 100; numeri dove la quota è almeno il 4%).
- secondo grafico: la percentuale sotto quel topic dell'articolo divisa per la percentuale sotto tutti gli articoli. Rosso = il tema dei commenti compare più del solito (×n dove è almeno il doppio), blu = meno del solito, grigio = meno di 30 commenti nella cella, lift non affidabile.

- **537 articoli e 114.483 commenti; 18 topic dell'articolo su 25** passano la regola del §6 (qui anche con il 2003-2016).
- **Ogni topic dell'articolo ha un tema "suo" nei commenti**, molto sovrarappresentato: bambini e famiglie → bambini e genitori (32% dei commenti, ×9); centri di detenzione → detenzione e prigioni (32%), più cure mediche (×6) e persone transgender in detenzione (×13); casi individuali di espulsione → il topic delle vicende personali ("mr, man, marijuana", 41%, ×12); raid nei luoghi di lavoro → lavoro e salari (41%, ×10) e datori di lavoro (×14); studenti e attivisti detenuti → Israele e libertà di parola (23%, ×21) e università (×11); proteste nelle città → Portland e manifestanti (28%, ×8); Corte Suprema → tribunali (24%, ×4).
- **Il topic dei commenti 0 ("agents, people, law": operato degli agenti, "abolish ICE") è il tema trasversale.** Pesa il 16% dei commenti dell'articolo medio e sale sotto quasi tutti gli articoli sull'azione di ICE: 49% sotto le sparatorie degli agenti, 25-26% sotto raid, finanziamento del DHS e proteste ai centri di detenzione, 23% sotto operazioni e arresti. Qualunque sia il fatto raccontato, una parte grande della discussione torna su cosa fanno gli agenti.
- **Primi segni di discussione su altri temi** (da verificare nel §7.2): sotto la politica nazionale crescono Trump e il potere (×3) e le menzogne (×3); sotto il finanziamento del DHS il muro (×6) e i tagli di bilancio (×8); sotto la guida del DHS Stephen Miller (×3); sotto le sparatorie polizia e maschere (×4). Sono temi vicini ma non coincidenti con quello dell'articolo.
- Il lift è alto anche per topic dei commenti piccoli: un ×20 su un tema di nicchia (Israele) pesa meno, in assoluto, di un ×3 su un tema grande. Le due mappe vanno lette insieme.

### 7.2 Quanto i commenti si allontanano dal loro articolo

Il §7.1 confronta due insiemi di topic nati da modelli diversi (25 per gli articoli, 45 per i commenti), quindi non dice quali topic dei commenti sono "lo stesso tema" dell'articolo. Questa sezione misura la distanza direttamente, senza abbinare i topic: articoli e commenti sono rappresentati con lo **stesso modello di embedding** (`all-mpnet-base-v2`), quindi la similarità coseno tra un commento e un articolo dice quanto parlano della stessa cosa.

**Misura.** Per ogni commento (≥ 10 parole, come nel §5) si calcola la similarità con il **proprio articolo** e con **tutti gli altri articoli ICE dello stesso periodo**. `quota_piu_vicini` è la quota degli altri articoli del periodo che sono più simili al commento del suo stesso articolo:
- vicino a 0: il commento parla proprio del suo articolo;
- intorno a 0,5: il suo articolo non gli è più vicino di un articolo ICE qualsiasi dello stesso periodo;
- vicino a 1: il commento somiglia di più ad altri articoli.

Un commento è **fuori tema** se `quota_piu_vicini ≥ 0,5`: almeno metà degli altri articoli del periodo gli sono più vicini del suo. Il confronto dentro lo stesso periodo serve a non penalizzare il 2025-presente, dove gli articoli hanno solo titolo e abstract (§5 del notebook 04: l'attacco manca dal 2025): anche gli articoli di confronto hanno lo stesso testo breve.

**Testo degli articoli.** Lo stesso del notebook 04: titolo + abstract + attacco (`semantic_text`). Gli embedding dei 2.093 articoli sono calcolati una volta e salvati in `articles_ice_embeddings_mpnet.npz`. Unità per i confronti tra topic: l'articolo, come nel §6 e nel §7.1 (quota di commenti fuori tema per articolo, poi media per topic).

In [ ]:
ARTICLE_EMBEDDINGS = PROCESSED_DIR / "articles_ice_embeddings_mpnet.npz"
OUT_COMMENT_SIMILARITY = PROCESSED_DIR / "comments_article_similarity.parquet"
OFF_TOPIC_THRESHOLD = 0.5
SIM_CHUNK = 20_000

# Testo degli articoli come nel notebook 04 (titolo + abstract + attacco)
arts = pd.read_parquet(IN_ARTICLE_TOPICS, columns=["id", "pub_date", "headline", "abstract", "lead_paragraph"])
arts["semantic_text"] = (arts["headline"].fillna("") + " " + arts["abstract"].fillna("") + " " + arts["lead_paragraph"].fillna("")
                         ).str.replace(r"\s+", " ", regex=True).str.strip()
arts["period"] = assign_period(pd.to_datetime(arts["pub_date"], utc=True).dt.year).astype(str)

cached = np.load(ARTICLE_EMBEDDINGS) if ARTICLE_EMBEDDINGS.exists() else None
if cached is None or cached["id"].tolist() != arts["id"].tolist():
    log.info("Embedding di %d articoli...", len(arts))
    vectors = embedding_model.encode(arts["semantic_text"].tolist(), batch_size=32, show_progress_bar=False)
    np.savez(ARTICLE_EMBEDDINGS, id=np.array(arts["id"].tolist(), dtype=str), embedding=vectors.astype(np.float32))
    cached = np.load(ARTICLE_EMBEDDINGS)
article_vectors = pd.DataFrame(cached["embedding"], index=cached["id"])


def normalize_rows(x: np.ndarray) -> np.ndarray:
    return x / np.linalg.norm(x, axis=1, keepdims=True)


# Commenti sopra soglia degli articoli del §7.1 (≥ 50 commenti, topic dell'articolo non outlier), con o senza topic
sim_comments = (eligible[["comment_id", "article_id"]]
                .merge(open_articles[["article_id"]], on="article_id")
                .merge(article_topics[["article_id", "article_topic"]], on="article_id"))
sim_comments = sim_comments[sim_comments["article_topic"] != -1].merge(arts[["id", "period"]].rename(columns={"id": "article_id"}), on="article_id")

parts = []
for period, group in sim_comments.groupby("period"):
    pool_ids = arts.loc[arts["period"] == period, "id"].tolist()       # tutti gli articoli ICE del periodo
    pool = normalize_rows(article_vectors.loc[pool_ids].values)
    position = {a: i for i, a in enumerate(pool_ids)}
    for start in range(0, len(group), SIM_CHUNK):
        chunk = group.iloc[start:start + SIM_CHUNK]
        sims = normalize_rows(all_embeddings.loc[chunk["comment_id"]].values) @ pool.T
        own = sims[np.arange(len(chunk)), chunk["article_id"].map(position).values]
        parts.append(pd.DataFrame({
            "comment_id": chunk["comment_id"].values,
            "sim_own": own,
            "sim_others_mean": (sims.sum(axis=1) - own) / (len(pool_ids) - 1),
            "quota_piu_vicini": (sims > own[:, None]).sum(axis=1) / (len(pool_ids) - 1),
        }))
comment_similarity = sim_comments.merge(pd.concat(parts), on="comment_id")
comment_similarity["fuori_tema"] = comment_similarity["quota_piu_vicini"] >= OFF_TOPIC_THRESHOLD
comment_similarity.to_parquet(OUT_COMMENT_SIMILARITY, index=False)
log.info("Scritto %s (%d commenti, %d articoli)", OUT_COMMENT_SIMILARITY.relative_to(BASE_DIR),
         len(comment_similarity), comment_similarity["article_id"].nunique())

# Controllo della misura: distribuzione di quota_piu_vicini per periodo
sim_periods = [q for q in period_labels if q in set(comment_similarity["period"])]
fig, ax = plt.subplots(figsize=(8, 3.8))
bins = np.linspace(0, 1, 41)
for color, q in zip(PERIOD_COLORS, sim_periods):
    values = comment_similarity.loc[comment_similarity["period"] == q, "quota_piu_vicini"]
    ax.hist(values, bins=bins, density=True, histtype="step", lw=1.6, color=color,
            label=f"{q}: {(values >= OFF_TOPIC_THRESHOLD).mean():.0%} off-topic")
ax.axvline(OFF_TOPIC_THRESHOLD, color="#555", ls="--", lw=1)
ax.set_xlabel("share of other articles of the period closer to the comment than its own article")
ax.set_ylabel("density")
ax.legend(frameon=False, fontsize=8)
ax.set_title("How close is each comment to its own article? (dashed = off-topic threshold)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "spillover_similarity_rank.png", dpi=200, bbox_inches="tight")
plt.show()

Questo grafico mostra quanto ogni commento è vicino al suo articolo, per ogni commento ci chiediamo se il suo articolo è quello che gli assomiglia di più o se ce ne sono altri più simili. Confrontiamo quindi il commento con il proprio articolo e con tutti gli altri dello stesso periodo (usando similarità tra embedding).
- asse x = percentuale deglio articoli del periodo che sono più simili al commento dell suo stesso articolo (0 = nessun articolo è più vicino, 0,5 = metà degli articoli sono più vicini, ...)
- asse y = densità, cioè quanti commenti cadono in ogni intervallo
- linea tratteggiata è la soglia del "fuori tema" (commenti a dx considerati fuori tema).

Notiamo che la maggior parte dei commenti parla del suo articolo (per il 36,5% dei ocmmenti il proprio articolo è tra il 2,5% di articoli più simili). L'11% dei commenti è fuori tema, inoltre la coda è piatta quindi chi cambia articolo non si sposta su un tema vicino ma varia. Le quattro curve quasi coincidono, dunque i risultati non sono influenzati dal fatto che dopo il 2025 abbiamo solo abstract perchè il confronto avviene sempre con articoli dello stesso periodo.

In [ ]:
# Quota di commenti fuori tema per articolo, poi media per topic dell'articolo con IC bootstrap sugli articoli
per_article_off = (comment_similarity.groupby(["article_id", "article_topic", "period"])["fuori_tema"].mean()
                   .rename("quota_fuori_tema").reset_index())
rng = np.random.default_rng(42)


def mean_ci(values: np.ndarray, n_boot: int = 2000) -> tuple[float, float, float]:
    boots = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    return values.mean(), *np.percentile(boots, [2.5, 97.5])


off_by_topic = pd.DataFrame(
    [(t, *mean_ci(g["quota_fuori_tema"].values)) for t, g in per_article_off.groupby("article_topic") if t in set(spill_topics)],
    columns=["article_topic", "media", "lo", "hi"]).set_index("article_topic").sort_values("media") * 100
off_by_period = pd.DataFrame(
    [(q, *mean_ci(per_article_off.loc[per_article_off["period"] == q, "quota_fuori_tema"].values)) for q in sim_periods],
    columns=["period", "media", "lo", "hi"]).set_index("period") * 100
overall_off = per_article_off["quota_fuori_tema"].mean() * 100

fig, (ax, axp) = plt.subplots(1, 2, figsize=(13, 0.36 * len(off_by_topic) + 1.8), gridspec_kw={"width_ratios": [2.4, 1]})
ypos = np.arange(len(off_by_topic))
ax.hlines(ypos, off_by_topic["lo"], off_by_topic["hi"], color=MAIN_COLOR, lw=1.3, alpha=0.6)
ax.scatter(off_by_topic["media"], ypos, color=MAIN_COLOR, s=34, zorder=3)
ax.axvline(overall_off, color="#555", ls="--", lw=1)
ax.set_yticks(ypos, [f"{topic_names[t]} (n={spill_selection.loc[t, 'articoli']})" for t in off_by_topic.index], fontsize=8)
ax.set_xlabel("% of the article's comments that are off-topic (mean over articles)")
ax.set_title("Off-topic comments by article topic (95% bootstrap CI; dashed = all articles)", loc="left", fontsize=10)

yp = np.arange(len(off_by_period))[::-1]
axp.hlines(yp, off_by_period["lo"], off_by_period["hi"], color=MAIN_COLOR, lw=1.3, alpha=0.6)
axp.scatter(off_by_period["media"], yp, color=MAIN_COLOR, s=34, zorder=3)
axp.axvline(overall_off, color="#555", ls="--", lw=1)
axp.set_yticks(yp, [f"{q}\n({(per_article_off['period'] == q).sum()} articles)" for q in off_by_period.index], fontsize=8)
axp.set_xlabel("% off-topic (mean over articles)")
axp.set_title("By period", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "spillover_offtopic_by_topic.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "spillover_offtopic_by_topic.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()


Nel pannello sinistro troviamo i topic sulle y e per ognuno un punto che rappresenta la percentuale media di commenti fuori tema per articolo (si calcola la percentuale dei suoi commenti fuori tema poi si fa la media sugli articoli del topic), la barra indica l intervallo di confidenza e la linea tratteggiata indica la media di tutti gli articoli (11%).

Nel pannello destro avviene lo stesso calcolo ma per periodo invece che per topic.

Possiamo vedere che **Shootings by agent** e **Raid and fear in communities** sono gli unici due topic a destra della media, sotto questi articoli i lettori parlano d'altro più spesso. Mentre per i topic che si trovano a sx della barra, i commenti parlano proprio di ciò che si affronta nell'articolo. I topic che attraversano la linea tratteggiata non si distinguono dalla media.

Notiamo che il topic delle sparatorie è quello con meno conversazione tra lettori (risultato precedente) e quello con più commenti fuori tema. Mentre per quanto riguarda i raid, hanno sia molta conversazione che commenti fuori tema, sono quindi articoli che fanno discuter ee spingono la conversazione verso altri temi.

### 7.3 Come evolve la discussione: temi di destinazione nelle catene di risposte

Ci poniamo tre domande sui commenti fuori tema della sezione precedente.

1. **Il cambio di tema nasce nelle conversazioni?** Quota di commenti fuori tema per livello della catena (1 = commento all'articolo, 2 = risposta diretta, 3+ = risposte a risposte). Media per articolo e poi tra articoli, come nella sezione §7.2; come controllo, il confronto **dentro lo stesso articolo** tra commenti di primo livello e risposte in catena, per escludere che la differenza dipenda da quali articoli hanno catene lunghe.
2. **Verso quali temi?** Per ogni topic dell'articolo, la distribuzione dei commenti fuori tema tra i topic dei commenti (`topic_merged`, solo commenti con un topic). Qui l'unità è il commento: per articolo i commenti fuori tema con un topic sono pochi e le quote per articolo sarebbero troppo instabili.
3. **Sintesi del §6 e del §7.** Per ogni topic dell'articolo presente in entrambe le sezioni: conversazione tra lettori (quota di commenti in catena, §6) contro discussione su altri temi (quota fuori tema, §7.2).

In [ ]:
# Commenti del §7.2 con la profondità nella catena; solo l'era con catene registrate
drift = comment_similarity.merge(comments[["comment_id", "chain_depth", "threaded"]], on="comment_id")
drift = drift[drift["threaded"]].copy()
drift["livello"] = drift["chain_depth"].clip(upper=5)
level_labels = {1: "1\ncomment on\nthe article", 2: "2\ndirect\nreply", 3: "3\nreply to\na reply", 4: "4", 5: "5+"}

per_article_level = drift.groupby(["article_id", "livello"])["fuori_tema"].mean().reset_index()
by_level = pd.DataFrame([(lv, *mean_ci(g["fuori_tema"].values), drift["livello"].eq(lv).sum())
                         for lv, g in per_article_level.groupby("livello")],
                        columns=["livello", "media", "lo", "hi", "commenti"]).set_index("livello")
by_level[["media", "lo", "hi"]] *= 100

# Controllo dentro lo stesso articolo: risposte in catena (livello >= 3) meno commenti di primo livello
MIN_PAIR_COMMENTS = 10
paired = drift.assign(gruppo=np.where(drift["livello"] >= 3, "catena", np.where(drift["livello"] == 1, "primo", None)))
paired = paired.dropna(subset=["gruppo"]).groupby(["article_id", "gruppo"])["fuori_tema"].agg(["mean", "size"]).unstack("gruppo")
paired = paired[(paired[("size", "catena")] >= MIN_PAIR_COMMENTS) & (paired[("size", "primo")] >= MIN_PAIR_COMMENTS)]
paired_diff = (paired[("mean", "catena")] - paired[("mean", "primo")]).values * 100
diff_mean, diff_lo, diff_hi = mean_ci(paired_diff)

fig, ax = plt.subplots(figsize=(8, 4.2))
x = np.arange(len(by_level))
ax.errorbar(x, by_level["media"], yerr=[by_level["media"] - by_level["lo"], by_level["hi"] - by_level["media"]],
            fmt="o-", color=MAIN_COLOR, capsize=3, lw=1.5)
for xi, row in zip(x, by_level.itertuples()):
    ax.text(xi, row.hi + 0.4, f"{row.commenti:,} comments", ha="center", fontsize=7, color="#555")
ax.set_xticks(x, [level_labels[lv] for lv in by_level.index], fontsize=8)
ax.set_ylabel("% off-topic (mean over articles)")
ax.set_ylim(0, by_level["hi"].max() + 3)
ax.set_title(f"Off-topic comments by position in the reply chain (95% bootstrap CI)\n"
             f"same article, reply chains minus top-level: {diff_mean:+.1f} points "
             f"[{diff_lo:+.1f}, {diff_hi:+.1f}], {len(paired_diff)} articles", loc="left", fontsize=9.5)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "spillover_by_chain_depth.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "spillover_by_chain_depth.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

Sull'asse delle x troviamo la profondità del gruppo di commenti nella catena di risposte, sulle y troviamo la percentuale degli articoli fuori tema (media per articolo, intervallo bootstrap al 95%); sopra ogni punto il numero di commenti. La seconda riga del titolo è un controllo: per ogni articolo calcoliamo quanti punti in più di commenti fuori tema hanno le risposte in catena (livello 3 e oltre) rispetto ai commenti di primo livello dello stesso articolo, poi facciamo la media tra gli articoli. Usiamo solo gli articoli con almeno 10 commenti in entrambi i gruppi. Così confrontiamo ogni articolo con sé stesso, e la differenza non può dipendere dal fatto che le catene lunghe nascono sotto articoli particolari.

- **Il cambio di tema nasce nelle conversazioni.** I commenti diretti all'articolo sono fuori tema nel 7,4% dei casi; le risposte dirette nel 14,6%, le risposte a risposte nel 17,9%, oltre il quarto livello nel 21,1%. Più la discussione si allunga, più si allontana dall'articolo.
- **Non dipende da quali articoli hanno catene lunghe.** 

In [ ]:
# Destinazioni: topic dei commenti dei commenti fuori tema, per topic dell'articolo (unità = commento)
TOP_DESTINATIONS = 3
dest = comment_similarity.merge(eligible[["comment_id", "topic_merged"]], on="comment_id")
dest = dest[(dest["topic_merged"] != -1) & dest["article_topic"].isin(spill_topics)]
off_dest = dest[dest["fuori_tema"]]
dest_share = pd.crosstab(off_dest["article_topic"], off_dest["topic_merged"], normalize="index") * 100
dest_rows = off_by_topic.index[::-1]                       # stesso ordine del §7.2: più fuori tema in alto
dest_cols = sorted({c for t in dest_rows for c in dest_share.loc[t].nlargest(TOP_DESTINATIONS).index},
                   key=lambda c: -dest_share[c].mean())
values = dest_share.loc[dest_rows, dest_cols].values
n_off = off_dest["article_topic"].value_counts()

fig, ax = plt.subplots(figsize=(0.55 * len(dest_cols) + 5, 0.36 * len(dest_rows) + 2.6))
im = ax.imshow(values, aspect="auto", cmap="Oranges", vmin=0)
for (r, c), v in np.ndenumerate(values):
    top = dest_cols[c] in dest_share.loc[dest_rows[r]].nlargest(TOP_DESTINATIONS).index
    if v >= 3 or top:
        ax.text(c, r, f"{v:.0f}", ha="center", va="center", fontsize=7,
                color="white" if v > values.max() / 2 else "#333", fontweight="bold" if top else "normal")
ax.set_yticks(range(len(dest_rows)),
              [f"{topic_names[t]} ({off_by_topic.loc[t, 'media']:.0f}% off-topic, {n_off.get(t, 0):,} comments)" for t in dest_rows],
              fontsize=7.5)
ax.set_xticks(range(len(dest_cols)), [f"{c}: {comment_topic_labels[c]}" for c in dest_cols], rotation=55, ha="right", fontsize=7.5)
ax.grid(False)
fig.colorbar(im, ax=ax, label="% of the off-topic comments", shrink=0.7, pad=0.01)
ax.set_title(f"Where off-topic comments go: comment topics of off-topic comments, by article topic\n"
             f"(bold = top {TOP_DESTINATIONS} for the row; columns = comment topics in some top {TOP_DESTINATIONS})",
             loc="left", fontsize=9.5)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "spillover_destinations.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "spillover_destinations.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()
log.info("Commenti fuori tema con un topic: %d su %d fuori tema (%.0f%%)", len(off_dest),
         comment_similarity["fuori_tema"].sum(), len(off_dest) / comment_similarity["fuori_tema"].sum() * 100)

Nelle righe troviamo i topic degli articoli, nello stesso ordine della sezione §7.2 (più fuori tema in alto; tra parentesi la quota fuori tema e il numero di commenti fuori tema con un topic). Le colonne corrispondono con i topic dei commenti che compaiono tra i primi 3 topic destinazione di almeno un topic dell'articolo. I valori nelle celle indicano quanti commenti relativi ad articoli con topic della riga, finiscono in quel topic (colonna).

- **Solo il 39% dei commenti fuori tema ha un topic** (10.848 su 27.841): gli altri sono outlier di HDBSCAN, coerente con l'idea che chi cambia argomento lo fa spesso con commenti generici.
- **La destinazione più comune è l'operato degli agenti** (topic 0, "abolish ICE"): un terzo dei commenti fuori tema sotto il finanziamento del DHS (34%), le proteste nelle città (34%) e la polizia locale (32%), un quinto sotto i casi individuali (23%) e la politica nazionale (21%). Da qualunque tema si parta, la discussione torna sul comportamento degli agenti.
- **Altre destinazioni sono specifiche:** dalle sparatorie e dalle proteste ai centri di detenzione si va verso Trump e i tribunali (topic 8: 15% e 22%); dai centri di detenzione e dai raid verso l'immigrazione illegale (16% e 11%); dalle deportazioni verso paesi terzi verso Biden e il confine (16%); dalla Corte Suprema verso lavoro e salari (17%); dalla guida del DHS verso bambini e famiglie (13%); dalla politica nazionale verso le proteste di Portland (9%).
- Il topic delle critiche al NYT (9) compare tra le destinazioni di molti topic (3-12%): parte della discussione fuori tema riguarda il giornale stesso.

In [ ]:
# Sintesi: conversazione (§6, media della quota in catena) contro discussione su altri temi (§7.2), topic presenti in entrambe
both = sorted(set(selected_topics) & set(off_by_topic.index))
synthesis = pd.DataFrame({
    "conversazione": sel.groupby("article_topic")["quota_in_catena"].mean().reindex(both) * 100,
    "fuori_tema": off_by_topic.loc[both, "media"],
    "articoli": spill_selection.loc[both, "articoli"],
})
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(synthesis["conversazione"], synthesis["fuori_tema"], s=synthesis["articoli"] * 6, color=MAIN_COLOR,
           alpha=0.6, edgecolor="white", linewidth=0.8)
# Spostamenti manuali delle etichette che si sovrappongono
SYNTH_OFFSETS = {"Local police and sanctuary cities": (-10, 0, "right"), "Deportations to third countries": (0, 10, "center"),
                 "Protests in cities": (8, -2, "left")}
for t, row in synthesis.iterrows():
    dx, dy, ha = SYNTH_OFFSETS.get(topic_names[t], (6, 0, "left"))
    ax.annotate(topic_names[t], (row["conversazione"], row["fuori_tema"]), xytext=(dx, dy), textcoords="offset points",
                fontsize=7.5, va="center", ha=ha)
pad_y = (synthesis["fuori_tema"].max() - synthesis["fuori_tema"].min()) * 0.12
ax.set_ylim(synthesis["fuori_tema"].min() - pad_y, synthesis["fuori_tema"].max() + pad_y)
ax.axvline(sel["quota_in_catena"].mean() * 100, color="#777", ls="--", lw=1)
ax.axhline(overall_off, color="#777", ls="--", lw=1)
corners = [(0.99, 0.97, "conversation and spillover", "right", "top"), (0.01, 0.97, "spillover without conversation", "left", "top"),
           (0.99, 0.02, "conversation on the article's topic", "right", "bottom"), (0.01, 0.02, "little of both", "left", "bottom")]
for x, y, text, ha, va in corners:
    ax.text(x, y, text, transform=ax.transAxes, ha=ha, va=va, fontsize=8, color="#777", style="italic")
ax.set_xlabel("% of comments in reply chains (§6, mean per article)")
ax.set_ylabel("% of off-topic comments (§7.2, mean per article)")
ax.set_title("Conversation vs spillover by article topic (bubble size = articles; dashed = all articles)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "spillover_vs_debate.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "spillover_vs_debate.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()
log.info("Spearman tra conversazione e fuori tema sui %d topic: %.2f", len(synthesis),
         synthesis["conversazione"].corr(synthesis["fuori_tema"], method="spearman"))

Questo grafico dice per ogni topic di un articolo quanto i lettori conversano tra loro e quanto escono dal tema. Ogni bolla rappresenta un topic dell'articolo (presente sia in sezione 6 che in 7.2), sulle X troviamo la percentuale media di commenti in catena (presi da 6), più si va a destra più il dibattito è acceso e profondo. Sulle Y troviamo la percentuale media dei commenti fuori tema (dalla sezione precedente), più in alto si va più la discussione va verso altri temi.
Bolle più grandi indicano più articoli per quel topic (le più piccole sono meno affidabili), la linea trattegiata indica invece le medie su tutti gli articoli e divide così il grafico in 4 quadranti.
- Alto a destra: la discussione è lunga e si allarga verso altri temi.
- Alto a sinistra: la conversazioni va verso altri temi ma senza un vero dibattito.
- Basso a destra: si discute molto del topic dell'articolo. 
- Basso a sinsitra: poca discussione e sullo stesso tema.

### 7.4 Stesso topic dell'articolo, periodi diversi: come cambiano i topic dei commenti

**Domanda.** Sotto articoli dello stesso tema ma di periodi diversi (per esempio i centri di detenzione nel 2003-2016, nel 2017-2020 e nel 2025-presente), i lettori parlano delle stesse cose? Il §7.1 mescola i periodi; qui si tiene fermo il topic dell'articolo e si guarda come cambia la discussione nel tempo.

**Topic dell'articolo analizzati**Servono topic presenti in più periodi con abbastanza articoli. Ne sono tenuti 8:
- con almeno 5 articoli in tre periodi: politica nazionale (2017-2020, 2021-2024, 2025+), centri di detenzione, polizia locale e città santuario, Corte Suprema (2003-2016, 2017-2020, 2025+);
- con almeno 5 articoli in due periodi (2017-2020 e 2025+) e presenti anche in un terzo: bambini e famiglie, operazioni e arresti, guida del DHS, raid e paura nelle comunità.

Gli altri topic esistono quasi solo nel 2025+ (sparatorie, proteste, Guardia Nazionale...) e non hanno un "prima" con cui confrontarsi.

**Misure.** Stessi dati del §7.1 (topic_merged articoli con ≥ 50 commenti, commenti con un topic) e stessa unità: il profilo di una cella è la media delle distribuzioni dei commenti dei suoi articoli tra i 45 topic dei commenti, ogni articolo pesa uno.
1. **Profili** (§7.4.1): heatmap dei topic dei commenti per ogni coppia (topic, periodo) con in fondo il profilo di tutti gli articoli di ciascun periodo come riferimento.
2. **Quanto cambia** (§7.4.2): distanza di Jensen-Shannon tra i profili di due periodi dello stesso topic (0 = stessa distribuzione, 1 = nessun topic dei commenti in comune). Con pochi articoli due profili differiscono anche per caso, quindi la distanza osservata è confrontata con quella ottenuta rimescolando 1.000 volte l'etichetta di periodo tra gli articoli del topic (test a permutazione).
3. **Cosa cambia** (§7.4.3): per il confronto 2017-2020 → 2025+, i 3 topic dei commenti che guadagnano e i 3 che perdono più quota, con intervallo bootstrap. Un rombo mostra quanto cambia lo stesso topic su tutti gli articoli, così si vede se il cambiamento è del tema o della discussione in generale.di tutta la discussione.


#### 7.4.1 Profili per periodo

In [ ]:
# Topic dell'articolo scelti dall'utente tra quelli presenti in più periodi (conteggi nella prima figura)
TIME_TOPICS = ["National politics: Trump and the parties", "Detention centers", "Local police and sanctuary cities",
               "Supreme Court and immigration policy", "Migrant children and families", "ICE operations and arrests",
               "Homeland Security leadership", "Raids and fear in communities"]
MIN_CELL_ARTICLES = 5
TOP_PER_CELL = 3

time_topic_ids = [topic_names.index[topic_names == name][0] for name in TIME_TOPICS]
article_period_of = spill_articles.set_index("article_id")["period"]
cell_articles = (pd.crosstab(article_topic_of, article_period_of)
                 .reindex(index=time_topic_ids, columns=period_labels, fill_value=0))


def cell_ids(topic, period) -> pd.Index:
    """Articoli di un topic dell'articolo in un periodo."""
    return article_topic_of.index[(article_topic_of == topic) & (article_period_of == period)]


# Profilo = distribuzione media dei commenti tra i topic dei commenti (ogni articolo pesa uno, come nel §7.1)
profiles = {(t, q): per_article_dist.loc[cell_ids(t, q)].mean()
            for t in time_topic_ids for q in period_labels if cell_articles.loc[t, q] >= MIN_CELL_ARTICLES}
period_profiles = {q: per_article_dist.loc[article_period_of.index[article_period_of == q]].mean() for q in period_labels}
log.info("Celle (topic, periodo) con almeno %d articoli: %d", MIN_CELL_ARTICLES, len(profiles))

rows = list(profiles) + [("all", q) for q in period_labels]
row_values = pd.DataFrame([profiles[r] if r[0] != "all" else period_profiles[r[1]] for r in rows]) * 100
cols = list(dict.fromkeys(c for r in profiles for c in profiles[r].nlargest(TOP_PER_CELL).index))
peak = row_values.iloc[:len(profiles)][cols].idxmax()
cols = sorted(cols, key=lambda c: (peak[c], -row_values.loc[peak[c], c]))
values = row_values[cols].values

fig, ax = plt.subplots(figsize=(0.5 * len(cols) + 6, 0.3 * len(rows) + 3))
im = ax.imshow(values, aspect="auto", cmap="Blues", vmin=0)
for (r, c), v in np.ndenumerate(values):
    if v >= 4:
        ax.text(c, r, f"{v:.0f}", ha="center", va="center", fontsize=6.5, color="white" if v > values.max() / 2 else "#333")
breaks = [i for i in range(1, len(rows)) if rows[i][0] != rows[i - 1][0]]
for b in breaks:
    ax.axhline(b - 0.5, color="#555", lw=1.2 if b == len(profiles) else 0.6)
ax.set_yticks(range(len(rows)), [f"{'All articles' if t == 'all' else topic_names[t]} · {q.replace('-presente', '+')}"
                                 f" ({cell_articles.loc[t, q] if t != 'all' else (article_period_of == q).sum()})"
                                 for t, q in rows], fontsize=7)
ax.set_xticks(range(len(cols)), [f"{c}: {comment_topic_labels[c]}" for c in cols], rotation=55, ha="right", fontsize=7)
ax.grid(False)
fig.colorbar(im, ax=ax, label="% of the comments of the mean article", shrink=0.6, pad=0.01)
ax.set_title(f"Comment topics under the same article topic in different periods (rows sum to 100 over all comment topics;\n"
             f"columns = top {TOP_PER_CELL} comment topics of some row; bottom block = all articles of the period)",
             loc="left", fontsize=9.5)
fig.savefig(PLOTS_DIR / "article_topic_time_profiles.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "article_topic_time_profiles.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

Una riga per coppia (topic dell'articolo, periodo), raggruppate per topic dell'articolo; tra parentesi gli articoli della cella. Le colonne sono i topic dei commenti che compaiono tra i primi 3 di almeno una riga, ordinati in modo che ogni colonna stia vicino alla riga in cui pesa di più. Ogni cella è la percentuale dei commenti dell'articolo medio della riga che appartiene a quel topic dei commenti (numeri dove è almeno il 4%); ogni riga somma a 100 su tutti i 45 topic dei commenti, qui ne sono mostrati 21. L'ultimo blocco è il profilo di tutti gli articoli del periodo: serve a capire se un cambiamento è proprio del tema o di tutta la discussione di quel periodo.

#### 7.4.2 Quanto cambia la discussione

In [ ]:
from itertools import combinations

from scipy.spatial.distance import jensenshannon

N_PERMUTATIONS = 1000


def js_distance(a: pd.Series, b: pd.Series) -> float:
    return float(jensenshannon(a.values, b.values, base=2))


# Per ogni coppia di periodi di uno stesso topic: distanza osservata e distanza attesa per caso
shift_rows = []
for t in time_topic_ids:
    periods_ok = [q for q in period_labels if (t, q) in profiles]
    for qa, qb in combinations(periods_ok, 2):
        ids_a, ids_b = cell_ids(t, qa), cell_ids(t, qb)
        pooled = per_article_dist.loc[ids_a.append(ids_b)].values
        null = np.empty(N_PERMUTATIONS)
        for i in range(N_PERMUTATIONS):
            perm = rng.permutation(len(pooled))
            null[i] = jensenshannon(pooled[perm[:len(ids_a)]].mean(axis=0), pooled[perm[len(ids_a):]].mean(axis=0), base=2)
        observed = js_distance(profiles[(t, qa)], profiles[(t, qb)])
        shift_rows.append({
            "article_topic": t, "da": qa, "a": qb, "articoli_da": len(ids_a), "articoli_a": len(ids_b),
            "distanza": observed, "caso_media": null.mean(), "caso_95": np.percentile(null, 95),
            "p": (np.sum(null >= observed) + 1) / (N_PERMUTATIONS + 1),
        })
shifts = pd.DataFrame(shift_rows)
shifts["eccesso"] = shifts["distanza"] - shifts["caso_media"]

fig, ax = plt.subplots(figsize=(9, 0.32 * len(shifts) + 1.8))
y = np.arange(len(shifts))[::-1]
ax.hlines(y, shifts["caso_media"], shifts["caso_95"], color=MUTED_COLOR, lw=5, label="expected by chance (mean to 95th pct.)")
ax.scatter(shifts["distanza"], y, s=36, zorder=3, color=np.where(shifts["p"] < 0.05, MAIN_COLOR, "white"), edgecolor=MAIN_COLOR)
ax.scatter([], [], s=36, color=MAIN_COLOR, label="observed, p < 0.05")
ax.scatter([], [], s=36, color="white", edgecolor=MAIN_COLOR, label="observed, p ≥ 0.05")
short = lambda q: q.replace("-presente", "+")
ax.set_yticks(y, [f"{topic_names[r.article_topic]}: {short(r.da)} → {short(r.a)} ({r.articoli_da}/{r.articoli_a})"
                  for r in shifts.itertuples()], fontsize=7.5)
for b in np.flatnonzero(shifts["article_topic"].values[1:] != shifts["article_topic"].values[:-1]):
    ax.axhline(y[b] - 0.5, color="#ccc", lw=0.6)
ax.set_xlabel("Jensen-Shannon distance between the comment-topic profiles of the two periods (0 = same, 1 = disjoint)")
ax.legend(frameon=False, fontsize=7.5, loc="lower right")
ax.set_title(f"How much the comment topics change between periods, same article topic\n"
             f"(chance = {N_PERMUTATIONS} permutations of the period label among the articles of the topic)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "article_topic_time_shift.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "article_topic_time_shift.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

Questo grafico risponde alla domanda: tra due periodi, sotto lo stesso tema dell'articolo i lettori parlano di cose diverse più di quanti ci si aspetterebbe per caso?

- per ogni riga c'è un confronto tra due periodi dello stesso topic dell'articolo (tra parentesi il numero di articoli).
- asse X: quanto sono diversi i due profili dei commenti (righe heatmap precedente), misurato con distanza di Jense-Shannon (1 = nessun topic dei commenti in comune, 0 = commenti distribuiti tra i topic allo stesso modo).
- punti blu sono la distanza osservata
- barra grigia è la distanza che si ottiene per caso
- se il punto è vuoto vuol dire che è compatibile con il caso.

Usiamo la barra del caso perchè anche due gruppi di articoli dello stesso periodo avrebbero profili diversi, perché ogni articolo ha la sua discussione. Con 6-15 articoli per gruppo la media ne risente molto. La distanza osservata quindi non va confrontata con lo zero: va confrontata con quella che si otterrebbe se il periodo non contasse nulla.

Come si calcola la barra. Esempio con "Detention centers 2003-2016 → 2025+" (12 e 14 articoli):
1. si mettono insieme i 26 articoli;
2. si mescolano e si dividono a caso in un gruppo da 12 e uno da 14, ignorando il periodo vero;
3. si calcola la distanza tra i due gruppi;
4. si ripete 1.000 volte.

Come leggerlo correttamente

- se il punto blu è lontano dalla barra grigia  vuol dire che sotto quel tema i lettori dei due periodi parlano davvero di cose diverse.
- se il punto è vuoto non possiamo dire che la discussione sia cambiata perchè quello che vedo si spiega già col fatto che ogni articolo ha la sua discussione.
- Il grafico dice se la discussione cambia, non cosa cambia. Per quello c'è il terzo grafico (guadagni e perdite) e la heatmap del §7.4.1.
- Punto vuoto non vuol dire "il tema non cambia": vuol dire che con questi pochi articoli non si vede un cambiamento più grande della variabilità tra articoli.
- Sono 16 test senza correzione per confronti multipli. I casi appena oltre la barra (raid, Corte Suprema 2017-2020 → 2025+) vanno presi come indicazioni. Quelli molto lontani (politica nazionale, Corte Suprema e polizia locale dal 2003-2016) sono solidi.
- Le distanze di topic diversi non vanno messe in classifica tra loro, perché la barra del caso cambia con il numero di articoli: conta quanto il punto si stacca dalla sua barra.

**"Per caso"** vuol dire: la differenza che vedresti anche se il periodo non contasse niente, solo perché ogni gruppo contiene articoli diversi.

L'idea

Prendi i 26 articoli sui centri di detenzione (12 del 2003-2016, 14 del 2025+). Ognuno ha la sua discussione:
- sotto un articolo su una morte in custodia i lettori parlano di cure mediche;
- sotto uno sulle condizioni delle celle parlano di prigioni;
- sotto uno su un singolo detenuto parlano della sua storia.

Se dividi questi 26 articoli in due gruppi qualsiasi, per esempio estraendoli a sorte, i due gruppi avranno quasi sempre profili un po' diversi. Non perché appartengono a epoche diverse, ma perché in un gruppo sono finiti più articoli sulle cure mediche e nell'altro più articoli sulle prigioni. Questa è la differenza "per caso": nasce solo da quali articoli capitano in quale gruppo.

Il p dice quante volte, su 1.000 estrazioni a sorte, è venuta fuori una distanza grande almeno quanto quella vera. Per esempio p = 0,02 vuol dire 20 volte su 1.000: è raro che il caso produca una differenza così, quindi il punto è pieno.

#### 7.4.3 Cosa cambia tra 2017-2020 e 2025-presente

In [ ]:
# Cosa cambia tra 2017-2020 e 2025-presente, l'unica coppia di periodi comune a tutti gli 8 topic
CHANGE_FROM, CHANGE_TO = "2017-2020", "2025-presente"
TOP_CHANGES = 3
N_BOOT = 2000


def boot_means(ids: pd.Index) -> np.ndarray:
    """Profili medi su N_BOOT ricampionamenti degli articoli (righe = ricampionamenti, colonne = topic dei commenti)."""
    x = per_article_dist.loc[ids].values
    return x[rng.integers(0, len(x), size=(N_BOOT, len(x)))].mean(axis=1)


general_change = (period_profiles[CHANGE_TO] - period_profiles[CHANGE_FROM]) * 100
change_rows = []
for t in time_topic_ids:
    diff = (profiles[(t, CHANGE_TO)] - profiles[(t, CHANGE_FROM)]) * 100
    boots = (boot_means(cell_ids(t, CHANGE_TO)) - boot_means(cell_ids(t, CHANGE_FROM))) * 100
    lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
    shown = list(diff.nlargest(TOP_CHANGES).index) + list(diff.nsmallest(TOP_CHANGES).index)
    for c in shown:
        k = diff.index.get_loc(c)
        change_rows.append({"article_topic": t, "comment_topic": c, "da": profiles[(t, CHANGE_FROM)][c] * 100,
                            "a": profiles[(t, CHANGE_TO)][c] * 100, "differenza": diff[c], "lo": lo[k], "hi": hi[k],
                            "differenza_generale": general_change[c]})
changes = pd.DataFrame(change_rows)

ncols = 2
nrows = int(np.ceil(len(time_topic_ids) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 2.3 * nrows + 0.8), sharex=True)
for ax, t in zip(axes.flat, time_topic_ids):
    g = changes[changes["article_topic"] == t].sort_values("differenza")
    y = np.arange(len(g))
    ax.barh(y, g["differenza"], color=np.where(g["differenza"] > 0, PERIOD_COLORS[1], MAIN_COLOR), alpha=0.85, height=0.6)
    ax.hlines(y, g["lo"], g["hi"], color="#333", lw=1)
    ax.scatter(g["differenza_generale"], y, marker="D", s=18, color="white", edgecolor="#333", zorder=3)
    ax.axvline(0, color="#555", lw=0.8)
    ax.set_yticks(y, [f"{c}: {comment_topic_labels[c]} ({a:.0f}→{b:.0f}%)" for c, a, b in zip(g["comment_topic"], g["da"], g["a"])],
                  fontsize=7.5)
    ax.set_title(f"{topic_names[t]} ({cell_articles.loc[t, CHANGE_FROM]} → {cell_articles.loc[t, CHANGE_TO]} articles)",
                 loc="left", fontsize=9)
for ax in axes.flat[len(time_topic_ids):]:
    ax.axis("off")
for ax in axes[-1]:
    ax.set_xlabel("change in share of comments, percentage points")
fig.suptitle(f"What changes from {CHANGE_FROM} to {CHANGE_TO.replace('-presente', '+')} under the same article topic: "
             f"top {TOP_CHANGES} gains (orange) and losses (blue)\n"
             f"line = 95% bootstrap CI over articles; diamond = change of the same comment topic over all articles",
             x=0.01, ha="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "article_topic_time_changes.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "article_topic_time_changes.png", dpi=200, bbox_inches="tight")  # copia per la tesi
plt.show()

Questo grafico dice di cosa si parla di più/meno sotto lo stesso tema dell'articolo nel peirodo post 2025 rispetto al 2017-2020 (il periodo 2021-23 non è abbastanza coperto da articoli).
- un pannello per ognuno degli 8 topic degli articoli (con numero di articoli per i due periodi). 
- ogni riga in un pannello rappresenta un topic dei commenti relativi a quell articolo (mostrati i 3 che guadagnano più quota e i 3 che ne perdono di più)
- la barra colorata indica la variazione della percentuale di commenti (nell'etichetta tra parentesi accanto al topic troviamo il prima e il dopo).
- la linea nera indica l'intervallo bootstrap (incertezza variazione dato il numero di articoli), se non attraversa lo 0 la variazione è affidabile altrimenti potrebbe essere una conseguenza della normale differenza tra articoli.
- il rombo bianco indica la variazione dello stesso topic dei commenti su tutti gli articoli dei due periodi (è il termine di paragone).

Tra il 17-20 e il 25+ cambia tutta la siscussione e non solo quella di un tema. Il rombo serve a distinguere: 
- barra e rombo vicini: il tema segue il cambiamento generale (non c'è niente di specifico)
- barra molto oltre il rombo: sotto questo tema il cambiamento è più forte che altrove.
- barra dal lato opposto del rombo: sotto questo tema va al contrario rispetto al resto.

Alcuni risultati

- **L'operato degli agenti (topic 0) cresce quasi ovunque, ma soprattutto per la deriva generale** (+10,7 punti su tutti gli articoli). Va oltre la deriva sotto i raid nelle comunità (12% → 28%) e la guida del DHS (1% → 14%), cioè dove gli articoli raccontano le azioni degli agenti. Sotto la politica nazionale va al contrario (20% → 10%): nel 2025+ sotto questi articoli si parla di Trump e dei partiti, non degli agenti.
- **Cambiamenti propri del tema:** sotto la Corte Suprema i tribunali raddoppiano (17% → 36%) e l'immigrazione illegale crolla (13% → 4%); sotto la polizia locale arrivano i tribunali (4% → 14%) ed escono l'immigrazione illegale (11% → 4%) e la California (6% → 1%); sotto la guida del DHS escono il muro con il Messico, il sindacato e i licenziamenti al DHS, Portland (temi del primo mandato Trump) ed entrano le vicende personali di donne (topic 12, 9% → 21%); sotto le operazioni e gli arresti escono patenti e documenti (6% → 1%) ed entrano i tagli di bilancio.
- **Bambini e famiglie:** il tema bambini e genitori scende di 16 punti (41% → 25%; l'intervallo attraversa di poco lo zero), più della deriva generale (−8,7); salgono le vicende personali di donne (4% → 15%). La separazione delle famiglie del 2018 lascia il posto alle storie di singole persone.
- **Immigrazione illegale (topic 3) cala sotto quasi tutti i temi** (Corte Suprema, polizia locale, raid, operazioni): nel 2025+ i lettori discutono meno della legittimità dell'immigrazione e più di cosa fanno ICE e i tribunali.
- Molti intervalli sono larghi (6-8 articoli nel 2017-2020): le variazioni grandi con intervallo che attraversa lo zero (detenzione e prigioni sotto i centri di detenzione, cure mediche, tribunali sotto la Corte Suprema) sono da leggere come tendenze.

**Risultato del §7.** Sotto gli articoli ICE i lettori parlano soprattutto del tema dell'articolo (89% dei commenti), e quando se ne allontanano lo fanno nelle conversazioni più che nel primo commento: la quota fuori tema quasi triplica dal primo livello alle catene lunghe. Il tema di arrivo più comune è l'operato degli agenti, qualunque sia il fatto raccontato. Gli articoli sulle sparatorie sono quelli in cui la discussione esce di più dal tema, quelli sui raid nelle comunità gli unici che uniscono conversazione e allargamento. Nel tempo, sotto lo stesso tema dell'articolo, la discussione resta simile dove il tema è un'azione concreta di ICE (centri di detenzione, operazioni e arresti) e cambia dove il tema è politico o giudiziario (politica nazionale, Corte Suprema, polizia locale): dal 2003-2016 al 2025+ si passa dalla legittimità dell'immigrazione ai tribunali e all'operato degli agenti. Limiti: la soglia del fuori tema (0,5) è una scelta; nel §7.4 le celle hanno 6-59 articoli e i confronti con il 2003-2016 riguardano solo 3 topic; il testo degli articoli è breve (titolo, abstract e, fino al 2024, attacco); i topic dei commenti coprono solo il 39% dei commenti fuori tema.